# Central Park Node Workflow — Adaptive Enclosing Rectangle Version

This notebook is a **Central Park-only** workflow.

It will:
1. Download the **walkable path network** for Central Park from OpenStreetMap using **OSMnx**.
2. Download the **Central Park boundary** from OSM.
3. Build a graph from the walkable path lines.
4. Automatically identify candidate nodes.
5. Compute an **adaptive enclosing rectangle** for the park.
6. Use that rectangle as the basis for grid generation.
7. Assign **grid-based codes** to nodes based on the adaptive rectangle grid.
8. Display the boundary, adaptive rectangle, grid, grid labels, and nodes on an interactive map.
9. Export all outputs.

Example grid-based code:
- `11001` means: rectangle-grid cell `(x=1, y=1)`, node `001` inside that cell.

Notes:
- Here the enclosing shape is constrained to be a **rectangle (or square as a special case)**.
- The rectangle is the park hull’s **minimum rotated rectangle**, which is the most stable enclosing rectangle for this use case.


## Install dependencies

Run this in Terminal if needed:

```bash
pip install geopandas shapely networkx pandas numpy pyproj osmnx folium matplotlib mapclassify
```


In [30]:
#pip install geopandas shapely networkx pandas numpy pyproj osmnx folium matplotlib mapclassify

In [31]:
from __future__ import annotations

import math
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import GeometryCollection, LineString, MultiLineString, Point, Polygon
from shapely.ops import substring, unary_union


def angle_between(v1, v2):
    x1, y1 = v1
    x2, y2 = v2
    n1 = math.hypot(x1, y1)
    n2 = math.hypot(x2, y2)
    if n1 == 0 or n2 == 0:
        return 0.0
    dot = max(-1.0, min(1.0, (x1 * x2 + y1 * y2) / (n1 * n2)))
    return float(math.degrees(math.acos(dot)))

def straight_distance(line):
    coords = list(line.coords)
    if len(coords) < 2:
        return 0.0
    return float(Point(coords[0]).distance(Point(coords[-1])))

def line_sinuosity(line):
    direct = straight_distance(line)
    if direct <= 1e-9:
        return 1.0
    return float(line.length / direct)

def cumulative_heading_change(line):
    coords = list(line.coords)
    if len(coords) < 3:
        return 0.0
    total = 0.0
    for i in range(1, len(coords) - 1):
        p0, p1, p2 = coords[i - 1], coords[i], coords[i + 1]
        v1 = (p1[0] - p0[0], p1[1] - p0[1])
        v2 = (p2[0] - p1[0], p2[1] - p1[1])
        total += angle_between(v1, v2)
    return float(total)

def representative_curvature(line):
    if float(line.length) <= 1e-9:
        return 0.0
    return float(cumulative_heading_change(line) / float(line.length))

def flatten_lines(geom):
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, LineString):
        return [geom] if len(geom.coords) >= 2 else []
    if isinstance(geom, MultiLineString):
        out = []
        for g in geom.geoms:
            out.extend(flatten_lines(g))
        return out
    if isinstance(geom, GeometryCollection):
        out = []
        for g in geom.geoms:
            out.extend(flatten_lines(g))
        return out
    return []

def ensure_wgs84(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    elif str(gdf.crs).upper() != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')
    return gdf

def project_to_local_metric(gdf):
    gdf = ensure_wgs84(gdf)
    utm = gdf.estimate_utm_crs()
    if utm is None:
        raise ValueError('Could not estimate local metric CRS.')
    return gdf.to_crs(utm), str(utm)

def round_point_key(pt, precision=3):
    return (round(float(pt.x), precision), round(float(pt.y), precision))

def fetch_osmnx_walk_network(place_name, output_dir, stem):
    import osmnx as ox
    G = ox.graph_from_place(place_name, network_type='walk', simplify=True)
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)
    output_dir.mkdir(parents=True, exist_ok=True)
    nodes_gdf.to_crs('EPSG:4326').to_file(output_dir / f'{stem}_nodes.geojson', driver='GeoJSON')
    edges_gdf.to_crs('EPSG:4326').to_file(output_dir / f'{stem}_edges.geojson', driver='GeoJSON')
    return G, nodes_gdf, edges_gdf

def fetch_park_boundary(place_name, output_dir, stem):
    import osmnx as ox
    boundary_gdf = ox.geocode_to_gdf(place_name)
    boundary_gdf = ensure_wgs84(boundary_gdf)
    output_dir.mkdir(parents=True, exist_ok=True)
    boundary_gdf.to_file(output_dir / f'{stem}_boundary.geojson', driver='GeoJSON')
    return boundary_gdf

def build_graph_from_lines(lines_metric, node_round_precision=3):
    unioned = unary_union(list(lines_metric.geometry))
    segments = flatten_lines(unioned)
    G = nx.Graph()
    coord_to_node = {}
    next_node_num = 1
    next_edge_num = 1

    def get_node_id(pt):
        nonlocal next_node_num
        key = round_point_key(pt, precision=node_round_precision)
        if key not in coord_to_node:
            node_id = f'N{next_node_num:04d}'
            coord_to_node[key] = node_id
            G.add_node(node_id, geometry=Point(key[0], key[1]))
            next_node_num += 1
        return coord_to_node[key]

    for seg in segments:
        if seg.length <= 1e-6 or len(seg.coords) < 2:
            continue
        u = get_node_id(Point(seg.coords[0]))
        v = get_node_id(Point(seg.coords[-1]))
        edge_id = f'E{next_edge_num:05d}'
        next_edge_num += 1
        attrs = {
            'edge_id': edge_id,
            'geometry': seg,
            'length_m': float(seg.length),
            'sinuosity': line_sinuosity(seg),
            'cum_turn_deg': cumulative_heading_change(seg),
            'curvature': representative_curvature(seg),
            'near_road_crossing': False,
            'has_stairs': False,
            'has_signalized_crossing': False,
            'surface': 'unknown',
        }
        if G.has_edge(u, v):
            if attrs['length_m'] < G[u][v]['length_m']:
                G[u][v].update(attrs)
        else:
            G.add_edge(u, v, **attrs)
    return G

def graph_nodes_to_gdf(G, crs):
    rows = []
    for node_id, attrs in G.nodes(data=True):
        degree = int(G.degree(node_id))
        node_type = 'junction' if degree >= 3 else ('dead_end' if degree == 1 else 'through_node')
        rows.append({
            'node_id': node_id,
            'node_type': node_type,
            'node_subtype': None,
            'degree': degree,
            'source': 'graph',
            'source_edge': None,
            'name': node_id,
            'geometry': attrs['geometry'],
        })
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)
    return gdf[gdf['node_type'].isin(['junction', 'dead_end'])].copy()

def intermediate_points_on_line(line, max_segment_len=80.0, max_turn_deg=45.0, min_spacing=20.0):
    out = []
    L = float(line.length)
    if L <= 1e-9:
        return out
    if L > max_segment_len:
        n_splits = int(L // max_segment_len)
        for i in range(1, n_splits + 1):
            d = min(i * max_segment_len, L - 1e-6)
            if 0 < d < L:
                out.append((line.interpolate(d), 'length_confirmation'))
    coords = list(line.coords)
    if len(coords) >= 3:
        accumulated_turn = 0.0
        last_added_distance = 0.0
        vertex_dist = [0.0]
        for i in range(1, len(coords)):
            vertex_dist.append(vertex_dist[-1] + Point(coords[i - 1]).distance(Point(coords[i])))
        for i in range(1, len(coords) - 1):
            p0, p1, p2 = coords[i - 1], coords[i], coords[i + 1]
            turn = angle_between((p1[0] - p0[0], p1[1] - p0[1]), (p2[0] - p1[0], p2[1] - p1[1]))
            accumulated_turn += abs(turn)
            d_here = vertex_dist[i]
            if accumulated_turn >= max_turn_deg and (d_here - last_added_distance) >= min_spacing:
                if min_spacing <= d_here <= max(L - min_spacing, min_spacing):
                    out.append((Point(p1), 'curve_confirmation'))
                    last_added_distance = d_here
                    accumulated_turn = 0.0
    return out

def extract_intermediate_candidates(G, crs, max_segment_len=80.0, max_turn_deg=45.0, min_spacing=20.0):
    rows = []
    idx = 1
    for _, _, data in G.edges(data=True):
        for pt, reason in intermediate_points_on_line(data['geometry'], max_segment_len=max_segment_len, max_turn_deg=max_turn_deg, min_spacing=min_spacing):
            rows.append({
                'node_id': f'C_INT_{idx:04d}',
                'node_type': 'intermediate_confirmation',
                'node_subtype': reason,
                'degree': None,
                'source': 'edge_rule',
                'source_edge': data['edge_id'],
                'name': f'{reason}_{idx}',
                'geometry': pt,
            })
            idx += 1
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)

def merge_close_candidate_nodes(structural_nodes, non_structural_nodes, merge_threshold=12.0):
    accepted_rows = []
    accepted_geoms = []
    for rec in structural_nodes.itertuples():
        accepted_rows.append({
            'node_id': rec.node_id,
            'node_type': rec.node_type,
            'node_subtype': rec.node_subtype,
            'degree': rec.degree,
            'source': rec.source,
            'source_edge': rec.source_edge,
            'name': rec.name,
            'notes': '',
            'geometry': rec.geometry,
        })
        accepted_geoms.append(rec.geometry)
    for rec in non_structural_nodes.itertuples():
        nearest_idx = None
        nearest_dist = None
        for i, g in enumerate(accepted_geoms):
            d = float(g.distance(rec.geometry))
            if nearest_dist is None or d < nearest_dist:
                nearest_idx = i
                nearest_dist = d
        if nearest_idx is not None and nearest_dist is not None and nearest_dist <= merge_threshold:
            tag = rec.node_subtype or rec.node_type
            if accepted_rows[nearest_idx]['notes']:
                if tag not in accepted_rows[nearest_idx]['notes']:
                    accepted_rows[nearest_idx]['notes'] += f'; {tag}'
            else:
                accepted_rows[nearest_idx]['notes'] = tag
        else:
            accepted_rows.append({
                'node_id': rec.node_id,
                'node_type': rec.node_type,
                'node_subtype': rec.node_subtype,
                'degree': rec.degree,
                'source': rec.source,
                'source_edge': rec.source_edge,
                'name': rec.name,
                'notes': '',
                'geometry': rec.geometry,
            })
            accepted_geoms.append(rec.geometry)
    return gpd.GeoDataFrame(accepted_rows, geometry='geometry', crs=structural_nodes.crs)

def make_adaptive_enclosing_rectangle(boundary_gdf):
    boundary_gdf = ensure_wgs84(boundary_gdf)
    boundary_metric, metric_crs = project_to_local_metric(boundary_gdf)
    hull_poly = unary_union(list(boundary_metric.geometry)).convex_hull
    rect_poly = hull_poly.minimum_rotated_rectangle
    rect_gdf_metric = gpd.GeoDataFrame(
        [{'geometry': rect_poly, 'rect_area': rect_poly.area, 'hull_area': hull_poly.area}],
        geometry='geometry',
        crs=metric_crs
    )
    rect_gdf = rect_gdf_metric.to_crs('EPSG:4326')
    extra_area = float(rect_poly.area - hull_poly.area)
    return rect_gdf, hull_poly, metric_crs, extra_area

def order_rect_corners(rect_poly_metric, hull_poly_metric):
    pts = np.asarray(list(rect_poly_metric.exterior.coords)[:-1], dtype=float)
    center = np.array(rect_poly_metric.centroid.coords[0], dtype=float)
    hull_coords = np.asarray(hull_poly_metric.exterior.coords[:-1], dtype=float)
    arr = hull_coords - center
    cov = np.cov(arr.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, np.argmax(eigvals)]
    minor = eigvecs[:, np.argmin(eigvals)]
    proj_major = (pts - center) @ major
    proj_minor = (pts - center) @ minor
    lower_idx = np.argsort(proj_minor)[:2]
    upper_idx = np.argsort(proj_minor)[2:]
    lower_idx = lower_idx[np.argsort(proj_major[lower_idx])]
    upper_idx = upper_idx[np.argsort(proj_major[upper_idx])]
    sw = pts[lower_idx[0]]
    se = pts[lower_idx[1]]
    nw = pts[upper_idx[0]]
    ne = pts[upper_idx[1]]
    return np.array([sw, se, ne, nw], dtype=float)

def _bilinear_rect_point(sw, se, ne, nw, s, t):
    return (
        (1 - s) * (1 - t) * sw
        + s * (1 - t) * se
        + s * t * ne
        + (1 - s) * t * nw
    )

def make_grid_from_rectangle(rect_gdf, hull_poly_metric, metric_crs, n_cols=5, n_rows=5):
    rect_metric = rect_gdf.to_crs(metric_crs)
    rect_poly_metric = rect_metric.geometry.iloc[0]
    sw, se, ne, nw = order_rect_corners(rect_poly_metric, hull_poly_metric)
    cells = []
    for ix in range(n_cols):
        for iy in range(n_rows):
            s0 = ix / n_cols
            s1 = (ix + 1) / n_cols
            t0 = iy / n_rows
            t1 = (iy + 1) / n_rows
            p00 = _bilinear_rect_point(sw, se, ne, nw, s0, t0)
            p10 = _bilinear_rect_point(sw, se, ne, nw, s1, t0)
            p11 = _bilinear_rect_point(sw, se, ne, nw, s1, t1)
            p01 = _bilinear_rect_point(sw, se, ne, nw, s0, t1)
            cell_poly = Polygon([tuple(p00), tuple(p10), tuple(p11), tuple(p01)])
            cells.append({'grid_x': ix + 1, 'grid_y': iy + 1, 'grid_id': f'{ix + 1}{iy + 1}', 'geometry': cell_poly})
    grid_metric = gpd.GeoDataFrame(cells, geometry='geometry', crs=metric_crs)
    return grid_metric.to_crs('EPSG:4326')

def assign_grid_codes_from_grid(nodes_gdf, grid_gdf, park_prefix=None):
    nodes = ensure_wgs84(nodes_gdf).copy()
    grid = ensure_wgs84(grid_gdf).copy()
    joined = gpd.sjoin(nodes, grid[['grid_x', 'grid_y', 'grid_id', 'geometry']], how='left', predicate='within')
    missing = joined['grid_x'].isna()
    if missing.any():
        nearest = gpd.sjoin_nearest(nodes.loc[missing].copy(), grid[['grid_x', 'grid_y', 'grid_id', 'geometry']], how='left', distance_col='grid_join_dist')
        for col in ['grid_x', 'grid_y', 'grid_id']:
            joined.loc[missing, col] = nearest[col].values
    joined['lat'] = joined.geometry.y
    joined['lon'] = joined.geometry.x
    joined = joined.sort_values(by=['grid_x', 'grid_y', 'lat', 'lon'], ascending=[True, True, False, True]).reset_index(drop=True)
    joined['cell_seq'] = joined.groupby(['grid_x', 'grid_y']).cumcount() + 1
    def make_code(row):
        xy = f"{int(row['grid_x'])}{int(row['grid_y'])}"
        seq = f"{int(row['cell_seq']):03d}"
        if park_prefix:
            return f"{park_prefix}-{xy}-{seq}"
        return f"{xy}{seq}"
    joined['grid_node_code'] = joined.apply(make_code, axis=1)
    return joined

def build_augmented_graph(base_G, final_nodes):
    extra_by_edge = {}
    base_edge_lookup = {}
    for u, v, data in base_G.edges(data=True):
        base_edge_lookup[data['edge_id']] = (u, v, data)
        extra_by_edge[data['edge_id']] = []
    for rec in final_nodes.itertuples():
        if rec.source_edge is None or rec.source_edge not in base_edge_lookup:
            continue
        u, v, data = base_edge_lookup[rec.source_edge]
        line = data['geometry']
        d = float(line.project(rec.geometry))
        if d <= 1e-6 or d >= line.length - 1e-6:
            continue
        if float(line.distance(rec.geometry)) > 2.0:
            continue
        extra_by_edge[rec.source_edge].append((d, rec.node_id))
    G2 = nx.Graph()
    for rec in final_nodes.itertuples():
        G2.add_node(rec.node_id, geometry=rec.geometry, node_type=rec.node_type, node_subtype=rec.node_subtype, name=rec.name, notes=getattr(rec, 'notes', ''))
    new_edge_num = 1
    for edge_id, (u, v, data) in base_edge_lookup.items():
        line = data['geometry']
        extra_nodes = sorted(extra_by_edge.get(edge_id, []), key=lambda x: x[0])
        distances = [0.0] + [d for d, _ in extra_nodes] + [float(line.length)]
        node_ids = [u] + [nid for _, nid in extra_nodes] + [v]
        for i in range(len(node_ids) - 1):
            d0, d1 = distances[i], distances[i + 1]
            if d1 - d0 <= 1e-6:
                continue
            seg = substring(line, d0, d1)
            if seg.is_empty or not isinstance(seg, LineString) or len(seg.coords) < 2:
                continue
            G2.add_edge(
                node_ids[i], node_ids[i + 1],
                edge_id=f'AE{new_edge_num:05d}',
                parent_edge_id=edge_id,
                geometry=seg,
                length_m=float(seg.length),
                sinuosity=line_sinuosity(seg),
                cum_turn_deg=cumulative_heading_change(seg),
                curvature=representative_curvature(seg),
                near_road_crossing=data.get('near_road_crossing', False),
                has_stairs=data.get('has_stairs', False),
                has_signalized_crossing=data.get('has_signalized_crossing', False),
                surface=data.get('surface', 'unknown'),
            )
            new_edge_num += 1
    return G2

def graph_edges_to_gdf(G, crs):
    rows = []
    for u, v, data in G.edges(data=True):
        rows.append({
            'u': u, 'v': v, 'edge_id': data['edge_id'], 'length_m': data['length_m'],
            'sinuosity': data['sinuosity'], 'cum_turn_deg': data['cum_turn_deg'],
            'curvature': data['curvature'], 'near_road_crossing': data.get('near_road_crossing', False),
            'has_stairs': data.get('has_stairs', False), 'has_signalized_crossing': data.get('has_signalized_crossing', False),
            'surface': data.get('surface', 'unknown'), 'geometry': data['geometry'],
        })
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=crs)

def export_outputs(output_dir, base_nodes_metric, final_nodes_metric, base_edges_metric, augmented_edges_metric):
    output_dir.mkdir(parents=True, exist_ok=True)
    base_nodes_metric.to_crs('EPSG:4326').to_file(output_dir / 'base_structural_nodes.geojson', driver='GeoJSON')
    final_nodes_metric.to_crs('EPSG:4326').to_file(output_dir / 'final_candidate_nodes.geojson', driver='GeoJSON')
    base_edges_metric.to_crs('EPSG:4326').to_file(output_dir / 'base_graph_edges.geojson', driver='GeoJSON')
    augmented_edges_metric.to_crs('EPSG:4326').to_file(output_dir / 'augmented_graph_edges.geojson', driver='GeoJSON')
    final_nodes_df = final_nodes_metric.to_crs('EPSG:4326').copy()
    final_nodes_df['lat'] = final_nodes_df.geometry.y
    final_nodes_df['lon'] = final_nodes_df.geometry.x
    final_nodes_df.drop(columns='geometry').to_csv(output_dir / 'final_candidate_nodes.csv', index=False)


## 1. Download the Central Park walkable network and boundary


In [33]:
# If needed, uncomment and run once:
# !pip install osmnx

from pathlib import Path

place_name = 'Central Park, Manhattan, New York, USA'
park_dir = Path('central_osmnx_outputs')

G_osm, osm_nodes_gdf, osm_edges_gdf = fetch_osmnx_walk_network(place_name, park_dir, 'central_park')
boundary_gdf = fetch_park_boundary(place_name, park_dir, 'central_park')

print('Downloaded Central Park walk network and boundary.')
print('Nodes:', G_osm.number_of_nodes())
print('Edges:', G_osm.number_of_edges())
print('Saved edges to:', park_dir / 'central_park_edges.geojson')
print('Saved boundary to:', park_dir / 'central_park_boundary.geojson')


/opt/anaconda3/lib/python3.12/site-packages/shapely/constructive.py:180: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(
/opt/anaconda3/lib/python3.12/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


Downloaded Central Park walk network and boundary.
Nodes: 1412
Edges: 4170
Saved edges to: central_osmnx_outputs/central_park_edges.geojson
Saved boundary to: central_osmnx_outputs/central_park_boundary.geojson


## 2. Inspect the walkable path file and boundary


In [35]:
from pathlib import Path
import geopandas as gpd

edges_path = Path('central_osmnx_outputs/central_park_edges.geojson')
boundary_path = Path('central_osmnx_outputs/central_park_boundary.geojson')

edges_raw = gpd.read_file(edges_path)
boundary_raw = gpd.read_file(boundary_path)

print('Edges geometry types:')
print(edges_raw.geometry.geom_type.value_counts(dropna=False))
print('\nBoundary geometry types:')
print(boundary_raw.geometry.geom_type.value_counts(dropna=False))


Edges geometry types:
LineString    4170
Name: count, dtype: int64

Boundary geometry types:
Polygon    1
Name: count, dtype: int64


/opt/anaconda3/lib/python3.12/site-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


## 3. Run node identification for Central Park


In [37]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

input_path = Path('central_osmnx_outputs/central_park_edges.geojson')
output_dir = Path('central_outputs')

raw = gpd.read_file(input_path)
if raw.crs is None:
    raw = raw.set_crs('EPSG:4326')
elif str(raw.crs).upper() != 'EPSG:4326':
    raw = raw.to_crs('EPSG:4326')

raw = raw[raw.geometry.notna()].copy()
raw = raw[~raw.geometry.is_empty].copy()

lines_wgs = raw[raw.geometry.geom_type.isin(['LineString', 'MultiLineString'])].copy()
lines_metric, metric_crs = project_to_local_metric(lines_wgs)

base_G = build_graph_from_lines(lines_metric)
structural_nodes = graph_nodes_to_gdf(base_G, metric_crs)

intermediate_nodes = extract_intermediate_candidates(
    base_G,
    crs=metric_crs,
    max_segment_len=80.0,
    max_turn_deg=45.0,
    min_spacing=20.0,
)

non_structural = gpd.GeoDataFrame(
    pd.concat([intermediate_nodes], ignore_index=True),
    geometry='geometry',
    crs=metric_crs,
)

final_nodes_metric = merge_close_candidate_nodes(
    structural_nodes,
    non_structural,
    merge_threshold=12.0,
)

augmented_G = build_augmented_graph(base_G, final_nodes_metric)
base_edges_metric = graph_edges_to_gdf(base_G, metric_crs)
augmented_edges_metric = graph_edges_to_gdf(augmented_G, metric_crs)

export_outputs(
    output_dir,
    structural_nodes,
    final_nodes_metric,
    base_edges_metric,
    augmented_edges_metric,
)

print('Done.')
print('Metric CRS:', metric_crs)
print('Base graph nodes:', base_G.number_of_nodes())
print('Base graph edges:', base_G.number_of_edges())
print('Structural nodes:', len(structural_nodes))
print('Intermediate nodes:', len(intermediate_nodes))
print('Final candidate nodes:', len(final_nodes_metric))


/opt/anaconda3/lib/python3.12/site-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/set_operations.py:421: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


Done.
Metric CRS: EPSG:32618
Base graph nodes: 13580
Base graph edges: 14300
Structural nodes: 1401
Intermediate nodes: 21
Final candidate nodes: 1418


## 4. Compute the adaptive enclosing rectangle


In [39]:
boundary_gdf = gpd.read_file('central_osmnx_outputs/central_park_boundary.geojson')
boundary_gdf = ensure_wgs84(boundary_gdf)

adaptive_rect_gdf, hull_poly_metric, rect_metric_crs, extra_area_metric = make_adaptive_enclosing_rectangle(boundary_gdf)

print('Adaptive rectangle computed.')
print('Extra area outside the park hull (square meters, approx):', extra_area_metric)

adaptive_rect_gdf


Adaptive rectangle computed.
Extra area outside the park hull (square meters, approx): 19830.649864084087


,geometry,rect_area,hull_area
0,"POLYGON ((-73.97299 40.76442, -73.94938 40.796...",3.452292e+06,3.432461e+06


## 5. Build a grid from the adaptive rectangle


In [41]:
# Change grid resolution here if needed
rect_grid_gdf = make_grid_from_rectangle(
    adaptive_rect_gdf,
    hull_poly_metric,
    rect_metric_crs,
    n_cols=5,
    n_rows=5
)

rect_grid_gdf.head()


,grid_x,grid_y,grid_id,geometry
0,1,1,11,"POLYGON ((-73.97299 40.76442, -73.96827 40.770..."
1,1,2,12,"POLYGON ((-73.97473 40.76516, -73.97001 40.771..."
2,1,3,13,"POLYGON ((-73.97647 40.76589, -73.97175 40.772..."
3,1,4,14,"POLYGON ((-73.97821 40.76662, -73.97349 40.773..."
4,1,5,15,"POLYGON ((-73.97995 40.76736, -73.97523 40.773..."


## 6. Assign grid-based node codes using the adaptive rectangle grid


In [43]:
grid_coded_nodes = assign_grid_codes_from_grid(final_nodes_metric, rect_grid_gdf, park_prefix=None)

print('Grid-coded nodes created.')
print(grid_coded_nodes[['node_id', 'grid_x', 'grid_y', 'cell_seq', 'grid_node_code']].head(20))


Grid-coded nodes created.
   node_id  grid_x  grid_y  cell_seq grid_node_code
0    N8478       1       1         1          11001
1    N3125       1       1         2          11002
2    N3124       1       1         3          11003
3    N1479       1       1         4          11004
4    N1489       1       1         5          11005
5    N1484       1       1         6          11006
6    N1466       1       1         7          11007
7    N1441       1       1         8          11008
8    N1520       1       1         9          11009
9    N1385       1       1        10          11010
10   N1368       1       1        11          11011
11   N1518       1       1        12          11012
12   N1427       1       1        13          11013
13   N1396       1       1        14          11014
14  N12979       1       1        15          11015
15   N1421       1       1        16          11016
16  N12980       1       1        17          11017
17   N1436       1       1        18  

## 7. Export the adaptive rectangle, rectangle grid, and grid-coded node table


In [45]:
grid_output_dir = Path('central_outputs_gridcoded')
grid_output_dir.mkdir(parents=True, exist_ok=True)

adaptive_rect_gdf.to_file(grid_output_dir / 'adaptive_rectangle.geojson', driver='GeoJSON')
rect_grid_gdf.to_file(grid_output_dir / 'adaptive_rectangle_grid.geojson', driver='GeoJSON')

grid_nodes_export = grid_coded_nodes.copy()
grid_nodes_export.to_file(grid_output_dir / 'final_candidate_nodes_gridcoded.geojson', driver='GeoJSON')
grid_nodes_export.drop(columns='geometry').to_csv(grid_output_dir / 'final_candidate_nodes_gridcoded.csv', index=False)

print('Saved adaptive rectangle outputs.')


Saved adaptive rectangle outputs.


## 8. Interactive map: boundary + adaptive rectangle + grid + nodes


In [47]:
# If explore() complains about missing packages, run:
# !pip install folium matplotlib mapclassify

m = boundary_gdf.explore(
    style_kwds={
        'fillOpacity': 0.02,
        'color': 'black',
        'weight': 2,
    },
    tooltip=[]
)

adaptive_rect_gdf.explore(
    m=m,
    style_kwds={
        'fillOpacity': 0.03,
        'color': 'green',
        'weight': 2,
    },
    tooltip=[]
)

rect_grid_gdf.explore(
    m=m,
    style_kwds={
        'fillOpacity': 0.03,
        'color': 'blue',
        'weight': 1,
    },
    tooltip=['grid_id', 'grid_x', 'grid_y']
)

grid_coded_nodes.to_crs('EPSG:4326').explore(
    m=m,
    color='red',
    markersize=4,
    tooltip=['grid_node_code', 'grid_x', 'grid_y', 'cell_seq', 'node_type']
)

m


## 9. Interactive map: boundary + rectangle + grid labels + nodes


In [49]:
import folium

grid_labels = rect_grid_gdf.copy()
grid_labels['label_pt'] = grid_labels.geometry.representative_point()

m = boundary_gdf.explore(
    style_kwds={
        'fillOpacity': 0.02,
        'color': 'black',
        'weight': 2,
    },
    tooltip=[]
)

adaptive_rect_gdf.explore(
    m=m,
    style_kwds={
        'fillOpacity': 0.03,
        'color': 'green',
        'weight': 2,
    },
    tooltip=[]
)

rect_grid_gdf.explore(
    m=m,
    style_kwds={
        'fillOpacity': 0.03,
        'color': 'blue',
        'weight': 1,
    },
    tooltip=['grid_id', 'grid_x', 'grid_y']
)

grid_coded_nodes.to_crs('EPSG:4326').explore(
    m=m,
    color='red',
    markersize=4,
    tooltip=['grid_node_code', 'grid_x', 'grid_y', 'cell_seq', 'node_type']
)

for _, row in grid_labels.iterrows():
    folium.Marker(
        location=[row['label_pt'].y, row['label_pt'].x],
        icon=folium.DivIcon(
            html=f"""
            <div style='font-size:10px; color:blue; font-weight:bold;'>
                {row['grid_id']}
            </div>
            """
        ),
    ).add_to(m)

m


## 10. Preview grid-coded nodes as a table


In [51]:
preview = grid_coded_nodes.to_crs('EPSG:4326').copy()
preview['lat'] = preview.geometry.y
preview['lon'] = preview.geometry.x
preview[['node_id', 'grid_node_code', 'grid_x', 'grid_y', 'cell_seq', 'node_type', 'node_subtype', 'notes', 'lat', 'lon']].head(30)


,node_id,grid_node_code,grid_x,grid_y,cell_seq,node_type,node_subtype,notes,lat,lon
0,N8478,11001,1,1,1,junction,None,,40.771523,-73.970002
1,N3125,11002,1,1,2,junction,None,,40.771477,-73.969936
2,N3124,11003,1,1,3,junction,None,,40.771421,-73.969869
3,N1479,11004,1,1,4,junction,None,,40.771361,-73.969798
4,N1489,11005,1,1,5,junction,None,,40.770988,-73.968713
5,N1484,11006,1,1,6,junction,None,,40.770604,-73.969326
6,N1466,11007,1,1,7,junction,None,,40.770392,-73.969860
7,N1441,11008,1,1,8,junction,None,,40.770184,-73.970025
8,N1520,11009,1,1,9,junction,None,,40.769996,-73.970978
9,N1385,11010,1,1,10,junction,None,,40.769971,-73.970939


## 11. Show exported files


In [53]:
print('central_outputs:')
for p in Path('central_outputs').glob('*'):
    print('  ', p.name)

print('\ncentral_outputs_gridcoded:')
for p in Path('central_outputs_gridcoded').glob('*'):
    print('  ', p.name)


central_outputs:
   augmented_graph_edges.geojson
   base_structural_nodes.geojson
   base_graph_edges.geojson
   final_candidate_nodes.geojson
   final_candidate_nodes.csv

central_outputs_gridcoded:
   adaptive_rectangle_grid.geojson
   adaptive_rectangle.geojson
   final_candidate_nodes_gridcoded.csv
   final_candidate_nodes_gridcoded.geojson


In [81]:
pip install ipyleaflet ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.7 MB/s eta 0:00:00 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [83]:
import math
import numpy as np
import geopandas as gpd
import networkx as nx

from scipy.spatial import cKDTree
from shapely.geometry import Point
from pyproj import Transformer

from ipyleaflet import (
    Map, Marker, Polyline, GeoJSON,
    LayersControl, WidgetControl
)
import ipywidgets as widgets


# ----------------------------
# 0. Basic assumptions
# ----------------------------
# Assumes these already exist in your notebook:
# - base_G: nx.Graph with node attr "geometry" in metric CRS
# - metric_crs: CRS string/object for base_G geometries

if "base_G" not in globals():
    raise ValueError("base_G not found. Run your graph-building cell first.")
if "metric_crs" not in globals():
    raise ValueError("metric_crs not found. Run your graph-building cell first.")


# ----------------------------
# 1. Graph -> GeoDataFrames
# ----------------------------
def graph_nodes_to_gdf_for_display(G, crs):
    rows = []
    for nid, attrs in G.nodes(data=True):
        rows.append({
            "node_id": nid,
            "geometry": attrs["geometry"]
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)

def graph_edges_to_gdf_for_display(G, crs):
    rows = []
    for u, v, data in G.edges(data=True):
        rows.append({
            "u": u,
            "v": v,
            "edge_id": data.get("edge_id"),
            "length_m": data.get("length_m", data["geometry"].length),
            "sinuosity": data.get("sinuosity", 1.0),
            "curvature": data.get("curvature", 0.0),
            "geometry": data["geometry"],
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)

nodes_metric = graph_nodes_to_gdf_for_display(base_G, metric_crs)
edges_metric = graph_edges_to_gdf_for_display(base_G, metric_crs)

nodes_wgs = nodes_metric.to_crs("EPSG:4326")
edges_wgs = edges_metric.to_crs("EPSG:4326")


# ----------------------------
# 2. Coordinate transformers
# ----------------------------
to_metric = Transformer.from_crs("EPSG:4326", metric_crs, always_xy=True)
to_wgs = Transformer.from_crs(metric_crs, "EPSG:4326", always_xy=True)


# ----------------------------
# 3. KD-tree for nearest-node snapping
# ----------------------------
node_ids = list(base_G.nodes())
node_xy = np.array([
    [base_G.nodes[n]["geometry"].x, base_G.nodes[n]["geometry"].y]
    for n in node_ids
], dtype=float)

node_tree = cKDTree(node_xy)

def snap_click_to_nearest_node(lat, lon):
    x, y = to_metric.transform(lon, lat)
    dist, idx = node_tree.query([x, y], k=1)
    snapped_node = node_ids[idx]
    snapped_geom = base_G.nodes[snapped_node]["geometry"]
    snapped_lon, snapped_lat = to_wgs.transform(snapped_geom.x, snapped_geom.y)
    return {
        "clicked_lat": lat,
        "clicked_lon": lon,
        "metric_x": x,
        "metric_y": y,
        "node_id": snapped_node,
        "snap_dist_m": float(dist),
        "snap_lat": float(snapped_lat),
        "snap_lon": float(snapped_lon),
    }


# ----------------------------
# 4. Route cost function
# ----------------------------
# This version strongly prefers short distance,
# but adds a penalty for more winding edges.
#
# You can tune alpha / beta later.
alpha_sinuosity = 25.0   # penalty for edges whose sinuosity > 1
beta_curvature = 150.0   # penalty for curved edges

def route_cost(u, v, d):
    length = float(d.get("length_m", d["geometry"].length))
    sinuosity = float(d.get("sinuosity", 1.0))
    curvature = float(d.get("curvature", 0.0))

    sinuosity_penalty = max(0.0, sinuosity - 1.0)
    curvature_penalty = max(0.0, curvature)

    return length + alpha_sinuosity * sinuosity_penalty + beta_curvature * curvature_penalty


# ----------------------------
# 5. A* heuristic
# ----------------------------
# Admissible enough here if your edge cost is nonnegative and
# still fundamentally length-dominated.
def heuristic_metric(n1, n2):
    p1 = base_G.nodes[n1]["geometry"]
    p2 = base_G.nodes[n2]["geometry"]
    return float(p1.distance(p2))


# ----------------------------
# 6. Convert path -> detailed lat/lon polyline
# ----------------------------
def edge_coords_in_path_order(G, u, v):
    line = G[u][v]["geometry"]
    coords = list(line.coords)

    pu = G.nodes[u]["geometry"]
    pv = G.nodes[v]["geometry"]

    start_dist = Point(coords[0]).distance(pu)
    end_dist = Point(coords[-1]).distance(pu)

    # If line is not oriented from u -> v, reverse it
    if start_dist <= end_dist:
        ordered = coords
    else:
        ordered = coords[::-1]

    return ordered

def path_to_latlon_coords(G, path):
    route_coords_metric = []

    for i in range(len(path) - 1):
        u = path[i]
        v = path[i + 1]
        edge_coords = edge_coords_in_path_order(G, u, v)

        # avoid duplicating shared vertices
        if i > 0:
            edge_coords = edge_coords[1:]

        route_coords_metric.extend(edge_coords)

    route_latlon = []
    for x, y in route_coords_metric:
        lon, lat = to_wgs.transform(x, y)
        route_latlon.append((lat, lon))

    return route_latlon

def route_stats(G, path):
    total_length = 0.0
    total_cost = 0.0
    total_sinuosity_penalty = 0.0
    total_curvature_penalty = 0.0

    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        d = G[u][v]

        length = float(d.get("length_m", d["geometry"].length))
        sinuosity = float(d.get("sinuosity", 1.0))
        curvature = float(d.get("curvature", 0.0))

        total_length += length
        total_cost += route_cost(u, v, d)
        total_sinuosity_penalty += max(0.0, sinuosity - 1.0)
        total_curvature_penalty += max(0.0, curvature)

    return {
        "length_m": total_length,
        "cost": total_cost,
        "sinuosity_penalty_sum": total_sinuosity_penalty,
        "curvature_penalty_sum": total_curvature_penalty,
        "num_nodes": len(path),
    }


# ----------------------------
# 7. Optional display layers
# ----------------------------
edge_layer = GeoJSON(
    data=edges_wgs.__geo_interface__,
    name="Walkable edges",
    style={
        "color": "#4C78A8",
        "weight": 2,
        "opacity": 0.6,
    }
)

# Only show candidate/final nodes if they already exist in notebook
candidate_layer = None
if "final_nodes_metric" in globals():
    final_nodes_wgs = final_nodes_metric.to_crs("EPSG:4326")
    candidate_layer = GeoJSON(
        data=final_nodes_wgs.__geo_interface__,
        name="Candidate nodes",
        point_style={
            "radius": 4,
            "color": "#E45756",
            "fillColor": "#E45756",
            "fillOpacity": 0.8,
        }
    )

In [85]:
from IPython.display import display

# ----------------------------
# 1. Map center
# ----------------------------
minx, miny, maxx, maxy = edges_wgs.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = Map(center=(center_lat, center_lon), zoom=15)

m.add(edge_layer)
if candidate_layer is not None:
    m.add(candidate_layer)

m.add(LayersControl(position="topright"))


# ----------------------------
# 2. UI widgets
# ----------------------------
status_html = widgets.HTML(
    value="<b>Instructions:</b> click once for start, click again for end. A third click starts a new route."
)
route_info_html = widgets.HTML(value="")
reset_btn = widgets.Button(description="Reset route", button_style="warning")

control_box = widgets.VBox([status_html, route_info_html, reset_btn])
m.add(WidgetControl(widget=control_box, position="topright"))


# ----------------------------
# 3. State
# ----------------------------
state = {
    "start": None,
    "end": None,
    "start_marker": None,
    "end_marker": None,
    "route_layer": None,
}

def remove_layer_safe(layer):
    if layer is not None and layer in m.layers:
        m.remove(layer)

def clear_route():
    remove_layer_safe(state["start_marker"])
    remove_layer_safe(state["end_marker"])
    remove_layer_safe(state["route_layer"])

    state["start"] = None
    state["end"] = None
    state["start_marker"] = None
    state["end_marker"] = None
    state["route_layer"] = None

    route_info_html.value = ""
    status_html.value = "<b>Instructions:</b> click once for start, click again for end. A third click starts a new route."

reset_btn.on_click(lambda btn: clear_route())


# ----------------------------
# 4. Route computation
# ----------------------------
def compute_and_draw_route():
    try:
        source = state["start"]["node_id"]
        target = state["end"]["node_id"]

        path = nx.astar_path(
            base_G,
            source=source,
            target=target,
            heuristic=heuristic_metric,
            weight=route_cost,
        )

        state["path"] = path

       
        route_latlon = path_to_latlon_coords(base_G, path)

        route_coords_metric = []

        total_length = 0.0
        total_cost = 0.0
        total_sinuosity_penalty = 0.0
        total_curvature_penalty = 0.0

        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            d = base_G[u][v]

            length = float(d.get("length_m", d["geometry"].length))
            sinuosity = float(d.get("sinuosity", 1.0))
            curvature = float(d.get("curvature", 0.0))

            total_length += length
            total_cost += route_cost(u, v, d)
            total_sinuosity_penalty += max(0.0, sinuosity - 1.0)
            total_curvature_penalty += max(0.0, curvature)

            edge_coords = edge_coords_in_path_order(base_G, u, v)
            if i > 0:
                edge_coords = edge_coords[1:]
            route_coords_metric.extend(edge_coords)

        route_line_metric = LineString(route_coords_metric)
        candidate_count = None
        hit_gdf = None

        if "final_nodes_metric" in globals():
            candidates_metric = final_nodes_metric.to_crs(metric_crs).copy()
            snap_tolerance_m = 8.0

            hit_mask = candidates_metric.geometry.distance(route_line_metric) <= snap_tolerance_m
            hit_gdf = candidates_metric[hit_mask].copy()
            candidate_count = len(hit_gdf)

        state["candidate_nodes_on_path_gdf"] = hit_gdf

        route_line = Polyline(
            locations=route_latlon,
            color="red",
            weight=5,
            fill=False,
            name="A* route"
        )

        remove_layer_safe(state["route_layer"])
        state["route_layer"] = route_line
        m.add(route_line)

        route_info_html.value = (
            f"<b>Route found</b><br>"
            f"start node: {source}<br>"
            f"end node: {target}<br>"
            f"route length: {total_length:.1f} m<br>"
            f"route cost: {total_cost:.1f}<br>"
            f"candidate nodes on path: {candidate_count}<br>"
            f"sinuosity penalty sum: {total_sinuosity_penalty:.3f}<br>"
            f"curvature penalty sum: {total_curvature_penalty:.3f}"
        )

        status_html.value = "<b>Done:</b> click again anywhere to start a new route."

    except nx.NetworkXNoPath:
        route_info_html.value = "<b>No path found.</b>"
        status_html.value = "<b>Try again:</b> click anywhere to start a new route."

# ----------------------------
# 5. Click handler
# ----------------------------
def handle_map_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs.get("coordinates")
    snapped = snap_click_to_nearest_node(lat, lon)

    # third click starts over
    if state["start"] is not None and state["end"] is not None:
        clear_route()

    marker = Marker(location=(snapped["snap_lat"], snapped["snap_lon"]))

    if state["start"] is None:
        state["start"] = snapped
        state["start_marker"] = marker
        m.add(marker)

        status_html.value = (
            f"<b>Start set</b><br>"
            f"clicked: ({lat:.6f}, {lon:.6f})<br>"
            f"snapped to node: {snapped['node_id']}<br>"
            f"snap distance: {snapped['snap_dist_m']:.1f} m<br>"
            f"Now click the destination."
        )

    elif state["end"] is None:
        state["end"] = snapped
        state["end_marker"] = marker
        m.add(marker)

        status_html.value = (
            f"<b>End set</b><br>"
            f"clicked: ({lat:.6f}, {lon:.6f})<br>"
            f"snapped to node: {snapped['node_id']}<br>"
            f"snap distance: {snapped['snap_dist_m']:.1f} m<br>"
            f"Computing A* route..."
        )

        compute_and_draw_route()

m.on_interaction(handle_map_click)

display(m)

Map(center=[40.782499099999995, -73.96564624999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### Supplementary import

In [87]:
import pandas as pd
import geopandas as gpd
import numpy as np
import networkx as nx
import re
from pathlib import Path

from shapely.geometry import Point, LineString

import folium

In [89]:
# pip install rapidfuzz gpxpy

In [93]:
from rapidfuzz import fuzz
import gpxpy
import gpxpy.gpx

### Define a more stable "nearest graph node snapping" function

In [95]:
def snap_latlon_to_nearest_graph_node(lat, lon):
    """
    Input WGS84 latitude and longitude, and return the nearest graph node and its snap information.
    Dependencies:
      - to_metric
      - node_tree
      - node_ids
      - base_G
      - to_wgs
    """
    x_m, y_m = to_metric.transform(lon, lat)
    dist_m, idx = node_tree.query([x_m, y_m])

    node_id = node_ids[idx]
    node_data = base_G.nodes[node_id]

    # 1) metric x/y
    if "x" in node_data and "y" in node_data:
        snap_x = float(node_data["x"])
        snap_y = float(node_data["y"])
        snap_lon, snap_lat = to_wgs.transform(snap_x, snap_y)

    # 2) geometry
    elif "geometry" in node_data and node_data["geometry"] is not None:
        geom = node_data["geometry"]

        snap_x = float(geom.x)
        snap_y = float(geom.y)
        snap_lon, snap_lat = to_wgs.transform(snap_x, snap_y)

    # 3) lon/lat
    elif "lon" in node_data and "lat" in node_data:
        snap_lon = float(node_data["lon"])
        snap_lat = float(node_data["lat"])
        snap_x, snap_y = to_metric.transform(snap_lon, snap_lat)

    else:
        raise KeyError(
            f"Node {node_id} has no usable coordinate fields. Available keys: {list(node_data.keys())}"
        )

    return {
        "node_id": node_id,
        "snap_dist_m": float(dist_m),
        "snap_lat": float(snap_lat),
        "snap_lon": float(snap_lon),
        "snap_x_m": float(snap_x),
        "snap_y_m": float(snap_y),
    }

### Encapsulate the starting point parsing function

In [97]:
def resolve_origin_from_gps(origin_lat, origin_lon):
    snapped = snap_latlon_to_nearest_graph_node(origin_lat, origin_lon)

    return {
        "origin_input_lat": float(origin_lat),
        "origin_input_lon": float(origin_lon),
        "origin_node_id": snapped["node_id"],
        "origin_snap_lat": snapped["snap_lat"],
        "origin_snap_lon": snapped["snap_lon"],
        "origin_snap_dist_m": snapped["snap_dist_m"],
        "origin_snap_x_m": snapped["snap_x_m"],
        "origin_snap_y_m": snapped["snap_y_m"],
    }

### Text normalization function

In [99]:
def normalize_text(s):
    if s is None:
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

### Destination fuzzy search function

In [101]:
def fuzzy_match_score(query, text):
    query_n = normalize_text(query)
    text_n = normalize_text(text)

    if not query_n or not text_n:
        return 0

    score_1 = fuzz.partial_ratio(query_n, text_n)
    score_2 = fuzz.token_sort_ratio(query_n, text_n)
    score_3 = fuzz.token_set_ratio(query_n, text_n)

    return max(score_1, score_2, score_3)

def search_destination_fuzzy(query, poi_table, top_k=5, min_score=45):
    tmp = poi_table.copy()
    tmp["match_score"] = tmp["search_text"].apply(lambda x: fuzzy_match_score(query, x))
    tmp = tmp.sort_values(["match_score", "display_name"], ascending=[False, True]).copy()
    tmp = tmp[tmp["match_score"] >= min_score].head(top_k)

    keep_cols = [
        c for c in [
            "node_id", "grid_node_code", "node_code", "node_type", "node_subtype",
            "category", "name", "label", "notes", "description",
            "display_name", "lat", "lon", "match_score"
        ] if c in tmp.columns
    ]

    return tmp[keep_cols].reset_index(drop=True)

### Destination resolution function

In [103]:
def resolve_destination_query(query, poi_table, top_k=5, ambiguity_gap=8):
    normalized_query = normalize_user_query(query)
    results = search_destination_fuzzy(normalized_query, poi_table, top_k=top_k)

    if len(results) == 0:
        raise ValueError(f"No destination match found for query: {query}")

    best = results.iloc[0].to_dict()

    ambiguous = False
    if len(results) >= 2:
        gap = float(results.iloc[0]["match_score"]) - float(results.iloc[1]["match_score"])
        if gap < ambiguity_gap:
            ambiguous = True

    return {
        "original_query": query,
        "normalized_query": normalized_query,
        "best_match": best,
        "all_candidates": results,
        "ambiguous": ambiguous,
    }

### Verify whether the destination node can be directly used in the graph

In [105]:
def validate_destination_node_id(dest_node_id, G):
    if dest_node_id not in G.nodes:
        raise ValueError(f"Destination node_id {dest_node_id} not found in graph.")
    return True

### Path calculation function

In [107]:
def compute_route_between_nodes(G, source_node, target_node):
    path = nx.astar_path(
        G,
        source=source_node,
        target=target_node,
        heuristic=heuristic_metric,
        weight=route_cost,
    )

    stats = route_stats(G, path)
    route_latlon = path_to_latlon_coords(G, path)

    return {
        "path_nodes": path,
        "route_latlon": route_latlon,
        "stats": stats,
    }

### GPX exported functions

In [109]:
def export_route_to_gpx(route_latlon, output_path, route_name="central_park_route"):
    gpx = gpxpy.gpx.GPX()

    track = gpxpy.gpx.GPXTrack(name=route_name)
    gpx.tracks.append(track)

    segment = gpxpy.gpx.GPXTrackSegment()
    track.segments.append(segment)

    for lat, lon in route_latlon:
        segment.points.append(
            gpxpy.gpx.GPXTrackPoint(latitude=float(lat), longitude=float(lon))
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(gpx.to_xml())

    return str(output_path)

### End-to-end main function

In [111]:
import re
from pathlib import Path
import pandas as pd

def route_from_gps_and_text(
    origin_lat,
    origin_lon,
    destination_query,
    poi_table,
    gpx_output_dir="gpx_outputs",
):
    # 1. Origin resolution
    origin_info = resolve_origin_from_gps(origin_lat, origin_lon)
    source_node = origin_info["origin_node_id"]

    # 2. Destination resolution
    dest_info = resolve_destination_query(destination_query, poi_table)
    best_match = dest_info["best_match"]

    # 3. Target node determination
    try:
        if "node_id" in best_match and pd.notna(best_match["node_id"]):
            target_node = best_match["node_id"]
            validate_destination_node_id(target_node, base_G)
            destination_node_resolution = "direct_node_id"
        else:
            raise ValueError("No usable node_id in destination table.")
    except Exception:
        target_node = snap_destination_row_to_graph(best_match)
        destination_node_resolution = "snapped_from_destination_latlon"

    # 4. Path calculation
    route_result = compute_route_between_nodes(base_G, source_node, target_node)

    # 5. Export GPX
    safe_query = re.sub(r"[^a-zA-Z0-9_]+", "_", destination_query.strip().lower())
    gpx_filename = f"route_to_{safe_query}.gpx"
    gpx_path = Path(gpx_output_dir) / gpx_filename

    route_name = f"Route to {best_match.get('display_name', 'destination')}"
    export_path = export_route_to_gpx(
        route_result["route_latlon"],
        gpx_path,
        route_name=route_name
    )

    # 6. Summary output
    result = {
        "origin": origin_info,
        "destination_query": destination_query,
        "destination_normalized_query": dest_info["normalized_query"],
        "destination_best_match": best_match,
        "destination_candidates": dest_info["all_candidates"],
        "destination_ambiguous": dest_info["ambiguous"],
        "destination_node_resolution": destination_node_resolution,
        "target_node_id": target_node,
        "route_stats": route_result["stats"],
        "path_nodes": route_result["path_nodes"],
        "route_latlon": route_result["route_latlon"],
        "gpx_path": export_path,
    }

    return result

### Find Central Park facilities from OSM

In [113]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np

In [114]:
park_place = "Central Park, Manhattan, New York, USA"
park_boundary = ox.geocode_to_gdf(park_place).to_crs("EPSG:4326")
park_boundary

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"POLYGON ((-73.98141 40.76846, -73.98135 40.768...",-73.981408,40.764727,-73.949606,40.800314,332679977,way,427818536,40.782773,-73.965363,leisure,park,24,0.612001,park,Central Park,"Central Park, New York County, New York, 11025..."


In [115]:
tags = {
    "amenity": ["toilets"],
    "leisure": ["playground"],
    "barrier": ["gate"],
    "entrance": True,
    "tourism": True,
}

In [116]:
raw_dest_gdf = ox.features_from_polygon(park_boundary.geometry.iloc[0], tags)
raw_dest_gdf.head()

/opt/anaconda3/lib/python3.12/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/set_operations.py:334: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


geometry  amenity  fee  \
element id                                                    
node    322546340  POINT (-73.97123 40.77352)  toilets   no   
        348454463   POINT (-73.9666 40.77388)  toilets  NaN   
        357620718  POINT (-73.96799 40.77146)      NaN  NaN   
        410224440  POINT (-73.95144 40.79716)  toilets  NaN   
        422438959  POINT (-73.97234 40.77014)      NaN  NaN   

                  toilets:handwashing wheelchair    architect  \
element id                                                      
node    322546340                 yes         no          NaN   
        348454463                 NaN        NaN          NaN   
        357620718                 NaN        NaN  Bruce Price   
        410224440                 NaN        yes          NaN   
        422438959                 NaN        NaN          NaN   

                             artist_name artwork_type  ele gnis:feature_id  \
element id                                                                   
node    322546340                    NaN          NaN  NaN             NaN   
        348454463                    NaN          NaN  NaN             NaN   
        357620718  Daniel Chester French         bust   26         2083525   
        410224440                    NaN          NaN  NaN             NaN   
        422438959            John Steell       statue  NaN             NaN   

                   ... internet_access internet_access:fee  \
element id         ...                                       
node    322546340  ...             NaN                 NaN   
        348454463  ...             NaN                 NaN   
        357620718  ...             NaN                 NaN   
        410224440  ...             NaN                 NaN   
        422438959  ...             NaN                 NaN   

                  internet_access:ssid name:he name:hi payment:cash  \
element id                                                            
node    322546340                  NaN     NaN     NaN          NaN   
        348454463                  NaN     NaN     NaN          NaN   
        357620718                  NaN     NaN     NaN          NaN   
        410224440                  NaN     NaN     NaN          NaN   
        422438959                  NaN     NaN     NaN          NaN   

                  payment:credit_cards payment:debit_cards ref:nrhp short_name  
element id                                                                      
node    322546340                  NaN                 NaN      NaN        NaN  
        348454463                  NaN                 NaN      NaN        NaN  
        357620718                  NaN                 NaN      NaN        NaN  
        410224440                  NaN                 NaN      NaN        NaN  
        422438959                  NaN                 NaN      NaN        NaN  

[5 rows x 137 columns]

In [117]:
dest_gdf = raw_dest_gdf.copy().reset_index()

dest_gdf["geometry"] = dest_gdf.geometry.representative_point()
dest_gdf = gpd.GeoDataFrame(dest_gdf, geometry="geometry", crs=raw_dest_gdf.crs)

dest_gdf["lat"] = dest_gdf.geometry.y
dest_gdf["lon"] = dest_gdf.geometry.x

dest_gdf.head()

,element,id,geometry,amenity,fee,toilets:handwashing,wheelchair,architect,artist_name,artwork_type,...,internet_access:ssid,name:he,name:hi,payment:cash,payment:credit_cards,payment:debit_cards,ref:nrhp,short_name,lat,lon
0,node,322546340,POINT (-73.97123 40.77352),toilets,no,yes,no,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.773523,-73.971226
1,node,348454463,POINT (-73.9666 40.77388),toilets,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.773883,-73.966597
2,node,357620718,POINT (-73.96799 40.77146),NaN,NaN,NaN,NaN,Bruce Price,Daniel Chester French,bust,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.771456,-73.967988
3,node,410224440,POINT (-73.95144 40.79716),toilets,NaN,NaN,yes,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.797161,-73.951437
4,node,422438959,POINT (-73.97234 40.77014),NaN,NaN,NaN,NaN,NaN,John Steell,statue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.770141,-73.972340


In [118]:
dest_gdf.columns.tolist()

['element',
 'id',
 'geometry',
 'amenity',
 'fee',
 'toilets:handwashing',
 'wheelchair',
 'architect',
 'artist_name',
 'artwork_type',
 'ele',
 'gnis:feature_id',
 'historic',
 'name',
 'start_date',
 'tourism',
 'website',
 'website:alternate',
 'wikidata',
 'wikipedia',
 'access',
 'artist:wikidata',
 'material',
 'subject:wikidata',
 'subject:wikipedia',
 'name:fr',
 'artist:wikipedia',
 'fountain',
 'name:en',
 'barrier',
 'colour',
 'memorial',
 'official_name',
 'subject',
 'attraction',
 'operator',
 'species:wikidata',
 'species:wikipedia',
 'unisex',
 'check_date',
 'information',
 'shop',
 'image',
 'source_ref',
 'subject:description',
 'height',
 'inscription_1',
 'inscription_2',
 'name:be',
 'name:pl',
 'name:ru',
 'statue',
 'support',
 'toilets:disposal',
 'artwork_subject',
 'year_of_construction',
 'inscription',
 'lit',
 'board_type',
 'note',
 'alt_name',
 'source',
 'opening_hours',
 'toilets:wheelchair',
 'male',
 'female',
 'wikimedia_commons',
 'direction',
 

In [119]:
def safe_get(row, col):
    return str(row[col]).strip() if col in row and pd.notna(row[col]) else ""

def build_dest_search_text(row):
    parts = []
    for col in [
        "name", "amenity", "leisure", "barrier", "entrance",
        "tourism", "operator", "brand", "description"
    ]:
        val = safe_get(row, col)
        if val:
            parts.append(val.lower())
    return " | ".join(parts)

def build_dest_display_name(row):
    parts = []
    for col in ["name", "amenity", "leisure", "barrier", "entrance", "tourism"]:
        val = safe_get(row, col)
        if val:
            parts.append(val)
    if not parts:
        return "unnamed_destination"
    return " | ".join(parts)

dest_gdf["search_text"] = dest_gdf.apply(build_dest_search_text, axis=1)
dest_gdf["display_name"] = dest_gdf.apply(build_dest_display_name, axis=1)

dest_gdf[["display_name", "search_text", "lat", "lon"]].head(20)

,display_name,search_text,lat,lon
0,toilets,toilets,40.773523,-73.971226
1,toilets,toilets,40.773883,-73.966597
2,Richard Morris Hunt | artwork,richard morris hunt | artwork,40.771456,-73.967988
3,toilets,toilets,40.797161,-73.951437
4,Sir Walter Scott | artwork,sir walter scott | artwork,40.770141,-73.972340
5,Robert Burns | artwork,robert burns | artwork,40.770193,-73.972566
6,Christopher Columbus | artwork,christopher columbus | artwork,40.769906,-73.972773
7,Three Dancing Maidens – Untermyer Fountain | f...,three dancing maidens – untermyer fountain | f...,40.794264,-73.951950
8,William Shakespeare | artwork,william shakespeare | artwork,40.769799,-73.972370
9,Indian Hunter | artwork,indian hunter | artwork,40.770377,-73.973152


In [120]:
for col in ["amenity", "leisure", "barrier", "entrance", "tourism", "name"]:
    if col in dest_gdf.columns:
        print("=" * 60)
        print(col)
        print(dest_gdf[col].value_counts(dropna=False).head(20))

amenity
amenity
NaN            148
toilets         24
fountain         3
clock            1
boat_rental      1
Name: count, dtype: int64
leisure
leisure
NaN           152
playground     24
slipway         1
Name: count, dtype: int64
barrier
barrier
NaN      157
gate      19
fence      1
Name: count, dtype: int64
entrance
entrance
NaN          163
yes            8
main           5
emergency      1
Name: count, dtype: int64
tourism
tourism
NaN            81
artwork        39
attraction     22
viewpoint      15
information    15
museum          2
zoo             2
picnic_site     1
Name: count, dtype: int64
name
name
NaN                                         77
Robert Burns                                 2
Tarr Family Playground                       2
East 110th Street Playground                 2
West 110th Street Playground                 2
Tisch's Children Zoo                         1
Charles A. Dana Discovery Center             1
Abraham and Joseph Spector Playground        1
Th

In [121]:
search_destination_fuzzy("toilets", dest_gdf, top_k=10)

,name,description,display_name,lat,lon,match_score
0,Tavern on the Green public restrooms,NaN,Tavern on the Green public restrooms | toilets,40.772275,-73.977263,100.0
1,NaN,NaN,toilets,40.773523,-73.971226,100.0
2,NaN,NaN,toilets,40.773883,-73.966597,100.0
3,NaN,NaN,toilets,40.797161,-73.951437,100.0
4,NaN,NaN,toilets,40.767432,-73.972162,100.0
5,NaN,NaN,toilets,40.767427,-73.971434,100.0
6,NaN,NaN,toilets,40.773489,-73.971079,100.0
7,NaN,NaN,toilets,40.775106,-73.968654,100.0
8,NaN,NaN,toilets,40.790333,-73.954617,100.0
9,NaN,NaN,toilets,40.780971,-73.961554,100.0


In [122]:
search_destination_fuzzy("playground", dest_gdf, top_k=10)

,name,description,display_name,lat,lon,match_score
0,Abraham and Joseph Spector Playground,NaN,Abraham and Joseph Spector Playground | playgr...,40.784518,-73.968414,100.0
1,Adventure Playground,NaN,Adventure Playground | playground,40.773315,-73.977503,100.0
2,Ancient Playground,NaN,Ancient Playground | playground,40.780809,-73.961661,100.0
3,Bernard Family Playground,NaN,Bernard Family Playground | playground,40.795757,-73.950384,100.0
4,Billy Johnson Playground,NaN,Billy Johnson Playground | playground,40.769478,-73.969923,100.0
5,Diana Ross Playground,NaN,Diana Ross Playground | playground,40.782288,-73.970885,100.0
6,East 110th Street Playground,NaN,East 110th Street Playground | playground,40.797713,-73.951952,100.0
7,East 110th Street Playground,NaN,East 110th Street Playground | playground,40.797573,-73.951729,100.0
8,East 72nd Street Playground,NaN,East 72nd Street Playground | playground,40.772108,-73.967814,100.0
9,Hecksher Playground,NaN,Hecksher Playground | playground | fence,40.768660,-73.977285,100.0


In [123]:
search_destination_fuzzy("gate", dest_gdf, top_k=10)

,name,description,display_name,lat,lon,match_score
0,Pioneers Gate,NaN,Pioneers Gate | gate,40.796759,-73.949806,100.0
1,The Emily Mumford Gate,NaN,The Emily Mumford Gate | gate,40.794516,-73.951768,100.0
2,Vanderbilt Gate,NaN,Vanderbilt Gate | gate,40.793567,-73.951869,100.0
3,NaN,NaN,gate,40.792743,-73.952781,100.0
4,NaN,NaN,gate,40.794518,-73.952229,100.0
5,NaN,NaN,gate,40.771676,-73.973389,100.0
6,NaN,NaN,gate,40.792042,-73.955122,100.0
7,NaN,NaN,gate,40.778591,-73.968060,100.0
8,NaN,NaN,gate,40.799858,-73.957060,100.0
9,NaN,NaN,gate,40.767577,-73.971642,100.0


In [124]:
search_destination_fuzzy("entrance", dest_gdf, top_k=10)

,name,description,display_name,lat,lon,match_score
0,The Dairy (Visitor Center and Gift Shop),NaN,The Dairy (Visitor Center and Gift Shop) | inf...,40.769097,-73.973707,75.000000
1,Central Park Carousel,NaN,Central Park Carousel | attraction,40.769943,-73.975248,71.428571
2,Central Park Zoo,NaN,Central Park Zoo | zoo,40.767601,-73.971845,71.428571
3,Charles A. Dana Discovery Center,NaN,Charles A. Dana Discovery Center | information,40.797081,-73.951457,66.666667
4,East 110th Street Playground,NaN,East 110th Street Playground | playground,40.797573,-73.951729,66.666667
5,Safari Playground,NaN,Safari Playground | playground,40.788462,-73.966419,66.666667
6,Tarr-Coyne Wild West Playground,NaN,Tarr-Coyne Wild West Playground | playground,40.789769,-73.965477,66.666667
7,West 110th Street Playground,NaN,West 110th Street Playground | playground,40.799777,-73.956988,66.666667
8,NaN,NaN,toilets,40.793719,-73.953084,66.666667
9,NaN,NaN,toilets,40.794196,-73.952736,66.666667


In [125]:
def normalize_user_query(query):
    q = query.lower().strip()

    alias_map = {
        "restroom": "toilets",
        "bathroom": "toilets",
        "wc": "toilets",
        "gate": "gate",
        "entrance": "gate",
        "exit": "gate",
        "play area": "playground",
    }

    return alias_map.get(q, q)

In [126]:
def snap_destination_row_to_graph(row):
    lat = float(row["lat"])
    lon = float(row["lon"])
    snapped = snap_latlon_to_nearest_graph_node(lat, lon)
    return snapped["node_id"]

### Use folium to display routes

In [128]:
def show_route_on_folium(result, zoom_start=16):
    origin = result["origin"]
    dest = result["destination_best_match"]
    route_latlon = result["route_latlon"]

    center_lat = route_latlon[0][0]
    center_lon = route_latlon[0][1]

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    # Original input starting point
    folium.CircleMarker(
        location=[origin["origin_input_lat"], origin["origin_input_lon"]],
        radius=5,
        popup="Original GPS origin",
        tooltip="Original GPS origin",
    ).add_to(m)

    # Snapped starting point
    folium.Marker(
        location=[origin["origin_snap_lat"], origin["origin_snap_lon"]],
        popup=f"Origin node: {origin['origin_node_id']}\nSnap dist: {origin['origin_snap_dist_m']:.1f} m",
        tooltip="Snapped origin",
    ).add_to(m)

    # End point
    folium.Marker(
        location=[dest["lat"], dest["lon"]],
        popup=dest["display_name"],
        tooltip="Destination",
    ).add_to(m)

    # Route
    folium.PolyLine(
        locations=route_latlon,
        weight=5,
        color="red",
        opacity=0.85,
    ).add_to(m)

    return m

### Test

In [130]:
demo_queries = ["toilets", "playground", "gate"]

demo_results = {}
for q in demo_queries:
    r = route_from_gps_and_text(
        origin_lat=40.7812,
        origin_lon=-73.9665,
        destination_query=q,
        poi_table=dest_gdf,
    )
    demo_results[q] = r
    print("=" * 70)
    print("QUERY:", q)
    print("BEST MATCH:", r["destination_best_match"]["display_name"])
    print("ROUTE LENGTH (m):", round(r["route_stats"]["length_m"], 1))
    print("GPX:", r["gpx_path"])

QUERY: toilets
BEST MATCH: Tavern on the Green public restrooms | toilets
ROUTE LENGTH (m): 1645.9
GPX: gpx_outputs/route_to_toilets.gpx
QUERY: playground
BEST MATCH: Abraham and Joseph Spector Playground | playground
ROUTE LENGTH (m): 571.1
GPX: gpx_outputs/route_to_playground.gpx
QUERY: gate
BEST MATCH: Pioneers Gate | gate
ROUTE LENGTH (m): 2500.2
GPX: gpx_outputs/route_to_gate.gpx


In [131]:
show_route_on_folium(demo_results["toilets"])

In [132]:
show_route_on_folium(demo_results["playground"])

In [133]:
show_route_on_folium(demo_results["gate"])

In [134]:
def preview_destination_candidates(query, poi_table, top_k=5):
    results = search_destination_fuzzy(normalize_user_query(query), poi_table, top_k=top_k)
    cols = [c for c in ["display_name", "lat", "lon", "match_score"] if c in results.columns]
    return results[cols]

In [135]:
preview_destination_candidates("gate", dest_gdf, top_k=10)

,display_name,lat,lon,match_score
0,Pioneers Gate | gate,40.796759,-73.949806,100.0
1,The Emily Mumford Gate | gate,40.794516,-73.951768,100.0
2,Vanderbilt Gate | gate,40.793567,-73.951869,100.0
3,gate,40.792743,-73.952781,100.0
4,gate,40.794518,-73.952229,100.0
5,gate,40.771676,-73.973389,100.0
6,gate,40.792042,-73.955122,100.0
7,gate,40.778591,-73.968060,100.0
8,gate,40.799858,-73.957060,100.0
9,gate,40.767577,-73.971642,100.0


In [136]:
preview_destination_candidates("72nd street", dest_gdf, top_k=10)
preview_destination_candidates("96th street", dest_gdf, top_k=10)

,display_name,lat,lon,match_score
0,East 110th Street Playground | playground,40.797713,-73.951952,81.818182
1,East 110th Street Playground | playground,40.797573,-73.951729,81.818182
2,West 110th Street Playground | playground,40.799775,-73.956986,81.818182
3,West 110th Street Playground | playground,40.799777,-73.956988,81.818182
4,East 72nd Street Playground | playground,40.772108,-73.967814,70.588235
5,7th Regiment Memorial | artwork,40.773762,-73.976440,63.636364
6,The Tempest | artwork,40.780448,-73.968645,63.636364
7,Three Dancing Maidens – Untermyer Fountain | f...,40.794264,-73.951950,62.500000
8,The Metropolitan Museum of Art | museum,40.779440,-73.963382,55.555556
9,Tavern on the Green public restrooms | toilets,40.772275,-73.977263,54.545455


In [137]:
rows = []
for q in ["toilets", "playground", "gate"]:
    r = route_from_gps_and_text(
        origin_lat=40.7812,
        origin_lon=-73.9665,
        destination_query=q,
        poi_table=dest_gdf,
    )
    rows.append({
        "query": q,
        "normalized_query": r["destination_normalized_query"],
        "best_match": r["destination_best_match"]["display_name"],
        "route_length_m": r["route_stats"]["length_m"],
        "target_node_id": r["target_node_id"],
        "gpx_path": r["gpx_path"],
    })

demo_df = pd.DataFrame(rows)
demo_df

,query,normalized_query,best_match,route_length_m,target_node_id,gpx_path
0,toilets,toilets,Tavern on the Green public restrooms | toilets,1645.900817,N9089,gpx_outputs/route_to_toilets.gpx
1,playground,playground,Abraham and Joseph Spector Playground | playgr...,571.057960,N5576,gpx_outputs/route_to_playground.gpx
2,gate,gate,Pioneers Gate | gate,2500.227966,N3775,gpx_outputs/route_to_gate.gpx


### AI Intergration 

In [174]:
import os
import json
import requests

In [177]:
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "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")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

OPENROUTER_MODEL = "openai/gpt-4o-mini"   # 你也可以换成自己账户里能用的模型

In [179]:
def call_openrouter_json(messages, model=OPENROUTER_MODEL, temperature=0):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",   # optional
        "X-OpenRouter-Title": "Central Park AI Routing Notebook"  # optional
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "response_format": {
            "type": "json_object"
        }
    }

    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    return json.loads(content)

In [181]:
def build_ai_candidates(candidates_df):
    rows = []
    for idx, row in candidates_df.iterrows():
        rows.append({
            "candidate_id": str(idx),
            "display_name": row.get("display_name", ""),
            "name": row.get("name", ""),
            "description": row.get("description", ""),
            "search_text": row.get("search_text", ""),
            "lat": float(row.get("lat")),
            "lon": float(row.get("lon")),
            "match_score": float(row.get("match_score", 0.0))
        })
    return rows

In [183]:
def resolve_destination_with_ai(user_query, candidate_rows, park_name="Central Park"):
    system_prompt = """
You are helping with destination selection for accessible park navigation.

Return JSON only with this schema:
{
  "decision": "select" | "clarify" | "reject",
  "selected_candidate_id": "",
  "clarification_question": "",
  "reason": ""
}

Rules:
- Use "select" if one candidate clearly matches the query.
- Use "clarify" if multiple candidates are plausible.
- Use "reject" if the query is unsupported or unrelated.
- Prefer semantic meaning over exact keyword overlap.
"""

    user_prompt = f"""
Park: {park_name}

User query:
{user_query}

Candidate destinations:
{json.dumps(candidate_rows, ensure_ascii=False, indent=2)}

Examples:
- "bathroom", "restroom" -> toilets
- "exit", "entrance" -> gate / entrance
- "play area for kids" -> playground
"""

    result = call_openrouter_json(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return result

In [185]:
def resolve_destination_query_hybrid(user_query, poi_table, top_k=5, direct_threshold=95):
    candidates = search_destination_fuzzy(user_query, poi_table, top_k=top_k)

    if len(candidates) == 0:
        return {
            "status": "reject",
            "message": "No destination match found."
        }

    top_score = float(candidates.iloc[0]["match_score"])

    # 高分直接走原逻辑
    if top_score >= direct_threshold:
        return {
            "status": "selected",
            "selected_row": candidates.iloc[0],
            "method": "fuzzy_direct",
            "candidates": candidates
        }

    # 否则 AI 参与
    ai_candidates = build_ai_candidates(candidates)
    ai_result = resolve_destination_with_ai(user_query, ai_candidates)

    decision = ai_result.get("decision", "").strip().lower()

    if decision == "select":
        selected_id = ai_result.get("selected_candidate_id", "")
        try:
            selected_idx = int(selected_id)
        except:
            return {
                "status": "reject",
                "message": f"AI returned invalid candidate id: {selected_id}",
                "ai_result": ai_result,
                "candidates": candidates
            }

        if selected_idx not in candidates.index:
            return {
                "status": "reject",
                "message": f"AI selected candidate not in top-k: {selected_idx}",
                "ai_result": ai_result,
                "candidates": candidates
            }

        return {
            "status": "selected",
            "selected_row": candidates.loc[selected_idx],
            "method": "ai_select",
            "ai_result": ai_result,
            "candidates": candidates
        }

    if decision == "clarify":
        return {
            "status": "clarify",
            "question": ai_result.get("clarification_question", "Which destination do you mean?"),
            "ai_result": ai_result,
            "candidates": candidates
        }

    return {
        "status": "reject",
        "message": ai_result.get("reason", "Unsupported destination query."),
        "ai_result": ai_result,
        "candidates": candidates
    }

In [187]:
def route_from_gps_and_text_ai(origin_lat, origin_lon, destination_query, poi_table):
    dest_result = resolve_destination_query_hybrid(destination_query, poi_table)

    if dest_result["status"] == "clarify":
        return {
            "status": "clarify",
            "question": dest_result["question"],
            "candidates_preview": dest_result["candidates"][
                ["display_name", "lat", "lon", "match_score"]
            ].to_dict(orient="records"),
            "dest_result": dest_result
        }

    if dest_result["status"] == "reject":
        return {
            "status": "reject",
            "message": dest_result["message"],
            "dest_result": dest_result
        }

    selected_row = dest_result["selected_row"]

    # 直接复用你原来已经跑通的函数
    r = route_from_gps_and_text(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        destination_query=selected_row["display_name"],
        poi_table=poi_table
    )

    r["ai_method"] = dest_result["method"]
    r["ai_selected_display_name"] = selected_row["display_name"]
    r["ai_selected_lat"] = float(selected_row["lat"])
    r["ai_selected_lon"] = float(selected_row["lon"])
    r["dest_result"] = dest_result

    return r

In [193]:
def assign_destination_category(row):
    amenity = str(row.get("amenity", "")).lower()
    leisure = str(row.get("leisure", "")).lower()
    barrier = str(row.get("barrier", "")).lower()
    entrance = str(row.get("entrance", "")).lower()
    tourism = str(row.get("tourism", "")).lower()
    name = str(row.get("name", "")).lower()
    display_name = str(row.get("display_name", "")).lower()
    search_text = str(row.get("search_text", "")).lower()

    text = " ".join([amenity, leisure, barrier, entrance, tourism, name, display_name, search_text])

    if "toilet" in text or "restroom" in text or amenity == "toilets":
        return "toilets"
    if "playground" in text or leisure == "playground":
        return "playground"
    if "gate" in text or barrier == "gate" or entrance in ["yes", "main", "emergency"]:
        return "gate"
    if "viewpoint" in text or tourism == "viewpoint":
        return "viewpoint"
    if tourism == "information":
        return "information"
    return "other"

dest_gdf["dest_category"] = dest_gdf.apply(assign_destination_category, axis=1)

dest_gdf["dest_category"].value_counts()

dest_category
other          66
gate           33
toilets        24
playground     24
viewpoint      15
information    15
Name: count, dtype: int64

In [195]:
def find_nearest_destination_by_category(origin_lat, origin_lon, poi_table, category, top_k=5):
    df = poi_table[poi_table["dest_category"] == category].copy()

    if df.empty:
        return None, None

    # 简单欧氏距离（经纬度），先够用
    df["dist_to_origin"] = ((df["lat"] - origin_lat) ** 2 + (df["lon"] - origin_lon) ** 2) ** 0.5
    df = df.sort_values("dist_to_origin", ascending=True)

    return df.iloc[0], df.head(top_k)

In [197]:
def resolve_query_to_category(query):
    q = query.lower().strip()

    if any(x in q for x in ["bathroom", "restroom", "toilet", "washroom"]):
        return {"status": "selected", "category": "toilets"}

    if any(x in q for x in ["playground", "play area", "kids", "children"]):
        return {"status": "selected", "category": "playground"}

    if any(x in q for x in ["exit", "entrance", "gate", "leave the park"]):
        return {"status": "selected", "category": "gate"}

    if any(x in q for x in ["view", "viewpoint", "scenic", "lookout"]):
        return {"status": "selected", "category": "viewpoint"}

    return {"status": "unclear", "category": None}

In [205]:
def route_from_gps_and_text_smart(origin_lat, origin_lon, destination_query, poi_table):
    cat_result = resolve_query_to_category(destination_query)

    if cat_result["status"] == "selected":
        category = cat_result["category"]

        best_row, nearby = find_nearest_destination_by_category(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            category=category,
            top_k=5
        )

        if best_row is None:
            return {
                "status": "reject",
                "message": f"No destinations found for category: {category}"
            }

        r = route_from_gps_and_text(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            destination_query=best_row["display_name"],
            poi_table=poi_table
        )

        # 关键：统一补上 status
        if "status" not in r:
            r["status"] = "ok"

        r["smart_category"] = category
        r["smart_selected_display_name"] = best_row["display_name"]
        r["smart_candidates"] = nearby[
            ["display_name", "lat", "lon", "dist_to_origin"]
        ].to_dict(orient="records")

        return r

    return {
        "status": "clarify",
        "question": "Could you specify whether you are looking for a restroom, playground, gate, or viewpoint?"
    }

In [207]:
demo_queries_smart = [
    "bathroom near me",
    "closest exit",
    "kids playground",
    "somewhere to see a view"
]

for q in demo_queries_smart:
    print("=" * 70)
    print("QUERY:", q)

    r = route_from_gps_and_text_smart(
        origin_lat=40.7812,
        origin_lon=-73.9665,
        destination_query=q,
        poi_table=dest_gdf
    )

    print("RAW KEYS:", list(r.keys()))
    print("STATUS:", r.get("status", "missing"))

    if r.get("status") == "ok":
        print("CATEGORY:", r.get("smart_category"))
        print("BEST MATCH:", r["destination_best_match"]["display_name"])
        print("ROUTE LENGTH (m):", round(r["route_stats"]["length_m"], 1))
    else:
        print("MESSAGE / QUESTION:", r.get("message", r.get("question", "")))

QUERY: bathroom near me
RAW KEYS: ['origin', 'destination_query', 'destination_normalized_query', 'destination_best_match', 'destination_candidates', 'destination_ambiguous', 'destination_node_resolution', 'target_node_id', 'route_stats', 'path_nodes', 'route_latlon', 'gpx_path', 'status', 'smart_category', 'smart_selected_display_name', 'smart_candidates']
STATUS: ok
CATEGORY: toilets
BEST MATCH: Tavern on the Green public restrooms | toilets
ROUTE LENGTH (m): 1645.9
QUERY: closest exit
RAW KEYS: ['origin', 'destination_query', 'destination_normalized_query', 'destination_best_match', 'destination_candidates', 'destination_ambiguous', 'destination_node_resolution', 'target_node_id', 'route_stats', 'path_nodes', 'route_latlon', 'gpx_path', 'status', 'smart_category', 'smart_selected_display_name', 'smart_candidates']
STATUS: ok
CATEGORY: gate
BEST MATCH: Pioneers Gate | gate
ROUTE LENGTH (m): 2500.2
QUERY: kids playground
RAW KEYS: ['origin', 'destination_query', 'destination_normalize

In [209]:
def inspect_category_candidates(origin_lat, origin_lon, category, poi_table, top_k=10):
    df = poi_table[poi_table["dest_category"] == category].copy()

    if df.empty:
        print("No candidates found.")
        return None

    # 当前你在用的“近”
    df["euclid_dist"] = ((df["lat"] - origin_lat) ** 2 + (df["lon"] - origin_lon) ** 2) ** 0.5
    df = df.sort_values("euclid_dist", ascending=True)

    cols = ["display_name", "lat", "lon", "euclid_dist"]
    return df[cols].head(top_k)

In [211]:
inspect_category_candidates(
    origin_lat=40.7812,
    origin_lon=-73.9665,
    category="gate",
    poi_table=dest_gdf,
    top_k=10
)

,display_name,lat,lon,euclid_dist
71,gate,40.778591,-73.968060,0.003040
109,yes,40.779298,-73.968896,0.003059
108,yes,40.779346,-73.968952,0.003074
96,gate,40.783405,-73.964347,0.003082
112,main,40.782188,-73.962520,0.004101
73,yes,40.778727,-73.962979,0.004303
74,main,40.779120,-73.962649,0.004376
72,main,40.779344,-73.962486,0.004422
127,gate,40.788432,-73.966257,0.007236
79,NYCT emergency exit | emergency,40.788631,-73.966707,0.007434


In [213]:
import pandas as pd

def compare_candidates_by_route(origin_lat, origin_lon, category, poi_table, top_k=8):
    """
    先在某个 category 内按直线距离筛前 top_k 个候选，
    再逐个调用现有 route_from_gps_and_text(...)，
    用真实 route length 排序。
    """
    df = poi_table[poi_table["dest_category"] == category].copy()

    if df.empty:
        return pd.DataFrame()

    # 先按直线距离筛一小批，避免全表都跑 route 太慢
    df["euclid_dist"] = ((df["lat"] - origin_lat) ** 2 + (df["lon"] - origin_lon) ** 2) ** 0.5
    df = df.sort_values("euclid_dist", ascending=True).head(top_k).copy()

    rows = []

    for idx, row in df.iterrows():
        try:
            r = route_from_gps_and_text(
                origin_lat=origin_lat,
                origin_lon=origin_lon,
                destination_query=row["display_name"],
                poi_table=poi_table
            )

            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "route_length_m": float(r["route_stats"]["length_m"]),
                "gpx_path": r.get("gpx_path", None)
            })

        except Exception as e:
            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "route_length_m": None,
                "gpx_path": None,
                "error": str(e)
            })

    out = pd.DataFrame(rows)

    if "route_length_m" in out.columns:
        out = out.sort_values("route_length_m", ascending=True, na_position="last")

    return out

In [215]:
def find_best_destination_by_route(origin_lat, origin_lon, poi_table, category, top_k=8):
    """
    在某个 category 内：
    1) 先按直线距离筛前 top_k 个候选
    2) 对每个候选跑真实 route
    3) 选 route_length 最短的那个作为 best
    """
    candidate_df = compare_candidates_by_route(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        category=category,
        poi_table=poi_table,
        top_k=top_k
    )

    if candidate_df.empty:
        return None, candidate_df

    valid_df = candidate_df[candidate_df["route_length_m"].notna()].copy()

    if valid_df.empty:
        return None, candidate_df

    best_idx = int(valid_df.iloc[0]["idx"])
    best_row = poi_table.loc[best_idx]

    return best_row, candidate_df

In [217]:
def resolve_query_to_category(query):
    q = query.lower().strip()

    if any(x in q for x in ["bathroom", "restroom", "toilet", "washroom"]):
        return {"status": "selected", "category": "toilets"}

    if any(x in q for x in ["playground", "play area", "kids", "children"]):
        return {"status": "selected", "category": "playground"}

    if any(x in q for x in ["exit", "entrance", "gate", "leave the park"]):
        return {"status": "selected", "category": "gate"}

    if any(x in q for x in ["view", "viewpoint", "scenic", "lookout"]):
        return {"status": "selected", "category": "viewpoint"}

    return {"status": "unclear", "category": None}

In [219]:
def route_from_gps_and_text_routeaware(origin_lat, origin_lon, destination_query, poi_table, top_k=8):
    """
    新逻辑：
    query -> category
    category -> 在该类目的地中比较多个候选的 route length
    选真实路线最短的那个
    再返回最终 route 结果
    """
    cat_result = resolve_query_to_category(destination_query)

    if cat_result["status"] == "selected":
        category = cat_result["category"]

        best_row, candidate_compare = find_best_destination_by_route(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            category=category,
            top_k=top_k
        )

        if best_row is None:
            return {
                "status": "reject",
                "message": f"No valid destinations found for category: {category}",
                "smart_category": category,
                "candidate_compare": candidate_compare.to_dict(orient="records") if isinstance(candidate_compare, pd.DataFrame) else []
            }

        # 用选出来的最佳 POI 再正式跑一次原始 routing
        r = route_from_gps_and_text(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            destination_query=best_row["display_name"],
            poi_table=poi_table
        )

        if "status" not in r:
            r["status"] = "ok"

        r["smart_category"] = category
        r["selection_method"] = "shortest_route_within_category"
        r["smart_selected_display_name"] = best_row["display_name"]
        r["candidate_compare"] = candidate_compare.to_dict(orient="records")

        return r

    return {
        "status": "clarify",
        "question": "Could you specify whether you are looking for a restroom, playground, gate, or viewpoint?"
    }

In [221]:
gate_compare = compare_candidates_by_route(
    origin_lat=40.7812,
    origin_lon=-73.9665,
    category="gate",
    poi_table=dest_gdf,
    top_k=8
)

gate_compare

,idx,display_name,lat,lon,euclid_dist,route_length_m,gpx_path
4,112,main,40.782188,-73.962520,0.004101,567.961164,gpx_outputs/route_to_main.gpx
6,74,main,40.779120,-73.962649,0.004376,567.961164,gpx_outputs/route_to_main.gpx
7,72,main,40.779344,-73.962486,0.004422,567.961164,gpx_outputs/route_to_main.gpx
1,109,yes,40.779298,-73.968896,0.003059,1941.913396,gpx_outputs/route_to_yes.gpx
2,108,yes,40.779346,-73.968952,0.003074,1941.913396,gpx_outputs/route_to_yes.gpx
5,73,yes,40.778727,-73.962979,0.004303,1941.913396,gpx_outputs/route_to_yes.gpx
0,71,gate,40.778591,-73.968060,0.003040,2500.227966,gpx_outputs/route_to_gate.gpx
3,96,gate,40.783405,-73.964347,0.003082,2500.227966,gpx_outputs/route_to_gate.gpx


In [223]:
demo_queries_routeaware = [
    "bathroom near me",
    "closest exit",
    "kids playground",
    "somewhere to see a view"
]

demo_results_routeaware = {}

for q in demo_queries_routeaware:
    print("=" * 70)
    print("QUERY:", q)

    r = route_from_gps_and_text_routeaware(
        origin_lat=40.7812,
        origin_lon=-73.9665,
        destination_query=q,
        poi_table=dest_gdf,
        top_k=8
    )

    demo_results_routeaware[q] = r

    print("STATUS:", r.get("status", "missing"))

    if r.get("status") == "ok":
        print("CATEGORY:", r.get("smart_category"))
        print("SELECTION METHOD:", r.get("selection_method"))
        print("BEST MATCH:", r["destination_best_match"]["display_name"])
        print("ROUTE LENGTH (m):", round(r["route_stats"]["length_m"], 1))
        print("GPX:", r.get("gpx_path", ""))
    else:
        print("MESSAGE / QUESTION:", r.get("message", r.get("question", "")))

QUERY: bathroom near me
STATUS: ok
CATEGORY: toilets
SELECTION METHOD: shortest_route_within_category
BEST MATCH: Tavern on the Green public restrooms | toilets
ROUTE LENGTH (m): 1645.9
GPX: gpx_outputs/route_to_toilets.gpx
QUERY: closest exit
STATUS: ok
CATEGORY: gate
SELECTION METHOD: shortest_route_within_category
BEST MATCH: main
ROUTE LENGTH (m): 568.0
GPX: gpx_outputs/route_to_main.gpx
QUERY: kids playground
STATUS: ok
CATEGORY: playground
SELECTION METHOD: shortest_route_within_category
BEST MATCH: Ancient Playground | playground
ROUTE LENGTH (m): 410.2
GPX: gpx_outputs/route_to_ancient_playground_playground.gpx
QUERY: somewhere to see a view
STATUS: ok
CATEGORY: viewpoint
SELECTION METHOD: shortest_route_within_category
BEST MATCH: slipway | viewpoint
ROUTE LENGTH (m): 950.5
GPX: gpx_outputs/route_to_slipway_viewpoint.gpx


In [225]:
query_to_inspect = "closest exit"

r_inspect = demo_results_routeaware[query_to_inspect]

if r_inspect.get("status") == "ok":
    compare_df = pd.DataFrame(r_inspect["candidate_compare"])
    display(compare_df)
else:
    print(r_inspect)

,idx,display_name,lat,lon,euclid_dist,route_length_m,gpx_path
0,112,main,40.782188,-73.962520,0.004101,567.961164,gpx_outputs/route_to_main.gpx
1,74,main,40.779120,-73.962649,0.004376,567.961164,gpx_outputs/route_to_main.gpx
2,72,main,40.779344,-73.962486,0.004422,567.961164,gpx_outputs/route_to_main.gpx
3,109,yes,40.779298,-73.968896,0.003059,1941.913396,gpx_outputs/route_to_yes.gpx
4,108,yes,40.779346,-73.968952,0.003074,1941.913396,gpx_outputs/route_to_yes.gpx
5,73,yes,40.778727,-73.962979,0.004303,1941.913396,gpx_outputs/route_to_yes.gpx
6,71,gate,40.778591,-73.968060,0.003040,2500.227966,gpx_outputs/route_to_gate.gpx
7,96,gate,40.783405,-73.964347,0.003082,2500.227966,gpx_outputs/route_to_gate.gpx


In [227]:
show_route_on_folium(demo_results_routeaware["bathroom near me"])

In [229]:
for q, r in demo_results_routeaware.items():
    print("=" * 80)
    print("QUERY:", q)

    if r.get("status") != "ok":
        print("NOT OK:", r)
        continue

    print("FINAL BEST MATCH:", r["destination_best_match"]["display_name"])
    compare_df = pd.DataFrame(r["candidate_compare"])

    if not compare_df.empty:
        display(compare_df[["display_name", "euclid_dist", "route_length_m"]])

QUERY: bathroom near me
FINAL BEST MATCH: Tavern on the Green public restrooms | toilets


,display_name,euclid_dist,route_length_m
0,toilets,0.002725,1645.900817
1,toilets,0.002840,1645.900817
2,toilets,0.003069,1645.900817
3,toilets,0.004951,1645.900817
4,toilets,0.006463,1645.900817
5,toilets,0.007318,1645.900817
6,toilets,0.008968,1645.900817
7,toilets,0.009015,1645.900817


QUERY: closest exit
FINAL BEST MATCH: main


,display_name,euclid_dist,route_length_m
0,main,0.004101,567.961164
1,main,0.004376,567.961164
2,main,0.004422,567.961164
3,yes,0.003059,1941.913396
4,yes,0.003074,1941.913396
5,yes,0.004303,1941.913396
6,gate,0.003040,2500.227966
7,gate,0.003082,2500.227966


QUERY: kids playground
FINAL BEST MATCH: Ancient Playground | playground


,display_name,euclid_dist,route_length_m
0,Ancient Playground | playground,0.004855,410.245765
1,Pinetum Playground | playground,0.002173,439.537121
2,Abraham and Joseph Spector Playground | playgr...,0.003831,571.057960
3,Toll Family Playground | playground,0.004249,638.822506
4,James Michael Levin Playground | playground,0.005567,652.134554
5,Diana Ross Playground | playground,0.004518,662.529742
6,Ruth and Arthur Smadbeck-Heckscher East Playgr...,0.004298,758.730020
7,Safari Playground | playground,0.007262,1058.040788


QUERY: somewhere to see a view
FINAL BEST MATCH: slipway | viewpoint


,display_name,euclid_dist,route_length_m
0,slipway | viewpoint,0.007168,950.534138
1,viewpoint,0.002215,1264.835121
2,viewpoint,0.003064,1264.835121
3,viewpoint,0.006088,1264.835121
4,viewpoint,0.006813,1264.835121
5,Skate Circle | viewpoint,0.010103,1264.835121
6,viewpoint,0.015774,1264.835121
7,viewpoint,0.016090,1264.835121


In [231]:
def call_openrouter_json_schema(messages, schema, model=OPENROUTER_MODEL, temperature=0):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "Central Park Multi-Step Navigation Demo"
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "trip_plan",
                "strict": True,
                "schema": schema
            }
        }
    }

    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
    resp.raise_for_status()
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    return json.loads(content)

In [233]:
def assign_destination_category_demo(row):
    text_parts = [
        str(row.get("amenity", "")).lower(),
        str(row.get("leisure", "")).lower(),
        str(row.get("barrier", "")).lower(),
        str(row.get("entrance", "")).lower(),
        str(row.get("tourism", "")).lower(),
        str(row.get("information", "")).lower(),
        str(row.get("name", "")).lower(),
        str(row.get("description", "")).lower(),
        str(row.get("display_name", "")).lower(),
        str(row.get("search_text", "")).lower(),
    ]
    text = " ".join(text_parts)

    if any(x in text for x in ["toilet", "restroom", "washroom"]) or str(row.get("amenity", "")).lower() == "toilets":
        return "toilets"

    if "playground" in text or str(row.get("leisure", "")).lower() == "playground":
        return "playground"

    if "gate" in text or str(row.get("barrier", "")).lower() == "gate" or str(row.get("entrance", "")).lower() in ["yes", "main", "emergency"]:
        return "gate"

    if "viewpoint" in text or "lookout" in text or "scenic" in text or str(row.get("tourism", "")).lower() == "viewpoint":
        return "viewpoint"

    return "other"

dest_gdf["dest_category"] = dest_gdf.apply(assign_destination_category_demo, axis=1)
dest_gdf["dest_category"].value_counts()

dest_category
other         81
gate          33
toilets       24
playground    24
viewpoint     15
Name: count, dtype: int64

In [235]:
def resolve_query_to_category_demo(query):
    q = query.lower().strip()

    if any(x in q for x in ["bathroom", "restroom", "toilet", "washroom"]):
        return {"status": "selected", "category": "toilets"}

    if any(x in q for x in ["playground", "play area", "kids", "children"]):
        return {"status": "selected", "category": "playground"}

    if any(x in q for x in ["exit", "entrance", "gate", "leave the park"]):
        return {"status": "selected", "category": "gate"}

    if any(x in q for x in ["view", "viewpoint", "scenic", "lookout"]):
        return {"status": "selected", "category": "viewpoint"}

    return {"status": "unclear", "category": None}

In [237]:
def compare_candidates_by_route(origin_lat, origin_lon, category, poi_table, top_k=8):
    df = poi_table[poi_table["dest_category"] == category].copy()

    if df.empty:
        return pd.DataFrame()

    # 先按直线距离筛一小批候选
    df["euclid_dist"] = ((df["lat"] - origin_lat) ** 2 + (df["lon"] - origin_lon) ** 2) ** 0.5
    df = df.sort_values("euclid_dist", ascending=True).head(top_k).copy()

    rows = []

    for idx, row in df.iterrows():
        try:
            r = route_from_gps_and_text(
                origin_lat=origin_lat,
                origin_lon=origin_lon,
                destination_query=row["display_name"],
                poi_table=poi_table
            )

            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "route_length_m": float(r["route_stats"]["length_m"]),
                "gpx_path": r.get("gpx_path", None)
            })

        except Exception as e:
            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "route_length_m": None,
                "gpx_path": None,
                "error": str(e)
            })

    out = pd.DataFrame(rows)

    if "route_length_m" in out.columns:
        out = out.sort_values("route_length_m", ascending=True, na_position="last")

    return out

In [239]:
def find_best_destination_by_route(origin_lat, origin_lon, poi_table, category, top_k=8):
    candidate_df = compare_candidates_by_route(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        category=category,
        poi_table=poi_table,
        top_k=top_k
    )

    if candidate_df.empty:
        return None, candidate_df

    valid_df = candidate_df[candidate_df["route_length_m"].notna()].copy()

    if valid_df.empty:
        return None, candidate_df

    best_idx = int(valid_df.iloc[0]["idx"])
    best_row = poi_table.loc[best_idx]

    return best_row, candidate_df

In [241]:
TRIP_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step_index": {"type": "integer"},
                    "user_intent": {"type": "string"},
                    "category": {
                        "type": "string",
                        "enum": ["viewpoint", "toilets", "gate", "playground", "unclear"]
                    },
                    "reason": {"type": "string"}
                },
                "required": ["step_index", "user_intent", "category", "reason"],
                "additionalProperties": False
            }
        }
    },
    "required": ["steps"],
    "additionalProperties": False
}

def ai_plan_trip_steps(user_request, park_name="Central Park"):
    system_prompt = """
You are helping decompose a park navigation request into ordered destination steps.

Allowed categories:
- viewpoint
- toilets
- gate
- playground
- unclear

Rules:
- "view", "scenic spot", "lookout", "viewpoint" -> viewpoint
- "bathroom", "restroom", "toilet" -> toilets
- "exit", "entrance", "leave the park" -> gate
- "playground", "kids area", "children's play area" -> playground

Return only the ordered steps as JSON.
"""

    user_prompt = f"""
Park: {park_name}
User request: {user_request}
"""

    return call_openrouter_json_schema(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        schema=TRIP_SCHEMA
    )

In [243]:
def route_one_step_routeaware(origin_lat, origin_lon, category, poi_table, top_k=8):
    best_row, candidate_compare = find_best_destination_by_route(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        poi_table=poi_table,
        category=category,
        top_k=top_k
    )

    if best_row is None:
        return {
            "status": "reject",
            "message": f"No valid destinations found for category: {category}",
            "category": category,
            "candidate_compare": candidate_compare.to_dict(orient="records") if isinstance(candidate_compare, pd.DataFrame) else []
        }

    r = route_from_gps_and_text(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        destination_query=best_row["display_name"],
        poi_table=poi_table
    )

    if "status" not in r:
        r["status"] = "ok"

    r["smart_category"] = category
    r["selection_method"] = "shortest_route_within_category"
    r["smart_selected_display_name"] = best_row["display_name"]
    r["candidate_compare"] = candidate_compare.to_dict(orient="records")

    return r

In [245]:
def run_ai_trip_routeaware(origin_lat, origin_lon, user_request, poi_table, top_k=8):
    ai_plan = ai_plan_trip_steps(user_request)
    steps = ai_plan["steps"]

    all_results = []
    current_lat = origin_lat
    current_lon = origin_lon

    for step in steps:
        category = step["category"]

        if category == "unclear":
            all_results.append({
                "step_index": step["step_index"],
                "user_intent": step["user_intent"],
                "category": category,
                "status": "clarify",
                "reason": step["reason"]
            })
            continue

        r = route_one_step_routeaware(
            origin_lat=current_lat,
            origin_lon=current_lon,
            category=category,
            poi_table=poi_table,
            top_k=top_k
        )

        all_results.append({
            "step_index": step["step_index"],
            "user_intent": step["user_intent"],
            "category": category,
            "ai_reason": step["reason"],
            "route_result": r
        })

        if r.get("status") == "ok":
            dest = r["destination_best_match"]
            current_lat = float(dest["lat"])
            current_lon = float(dest["lon"])

    return {
        "user_request": user_request,
        "ai_plan": ai_plan,
        "steps": all_results
    }

In [247]:
def summarize_ai_trip(ai_trip_result):
    rows = []

    for step in ai_trip_result["steps"]:
        rr = step.get("route_result", {})

        rows.append({
            "step_index": step["step_index"],
            "user_intent": step["user_intent"],
            "ai_category": step["category"],
            "ai_reason": step.get("ai_reason", step.get("reason", "")),
            "status": rr.get("status", step.get("status", "unknown")),
            "selected_destination": rr.get("smart_selected_display_name", None),
            "route_length_m": rr.get("route_stats", {}).get("length_m", None)
        })

    return pd.DataFrame(rows).sort_values("step_index")

In [249]:
def show_ai_trip_on_folium(ai_trip_result, origin_lat, origin_lon):
    color_cycle = ["red", "blue", "green", "purple", "orange"]
    m = folium.Map(location=[origin_lat, origin_lon], zoom_start=14)

    # 起点
    folium.Marker(
        [origin_lat, origin_lon],
        popup="Start",
        icon=folium.Icon(color="black", icon="play")
    ).add_to(m)

    step_num = 0

    for step in ai_trip_result["steps"]:
        rr = step.get("route_result", {})
        if rr.get("status") != "ok":
            continue

        step_num += 1
        color = color_cycle[(step_num - 1) % len(color_cycle)]

        route_latlon = rr["route_latlon"]
        folium.PolyLine(
            route_latlon,
            color=color,
            weight=6,
            opacity=0.85,
            popup=f"Step {step_num}: {step['user_intent']}"
        ).add_to(m)

        dest = rr["destination_best_match"]
        popup_html = f"""
        <b>Step {step_num}</b><br>
        Intent: {step['user_intent']}<br>
        AI category: {step['category']}<br>
        AI reason: {step['ai_reason']}<br>
        Selected: {rr['smart_selected_display_name']}<br>
        Route length: {round(rr['route_stats']['length_m'], 1)} m
        """

        folium.Marker(
            [dest["lat"], dest["lon"]],
            popup=folium.Popup(popup_html, max_width=350),
            icon=folium.Icon(color=color)
        ).add_to(m)

    return m

In [251]:
ai_trip_demo = run_ai_trip_routeaware(
    origin_lat=40.7812,
    origin_lon=-73.9665,
    user_request="go to a scenic spot, then to a restroom, then exit the park",
    poi_table=dest_gdf,
    top_k=8
)

summarize_ai_trip(ai_trip_demo)

,step_index,user_intent,ai_category,ai_reason,status,selected_destination,route_length_m
0,1,go to a scenic spot,viewpoint,User wants to visit a scenic area.,ok,slipway | viewpoint,950.534138
1,2,go to a restroom,toilets,User needs to use the bathroom.,ok,toilets,902.825431
2,3,exit the park,gate,User wants to leave the park.,ok,yes,599.776861


In [253]:
show_ai_trip_on_folium(
    ai_trip_demo,
    origin_lat=40.7812,
    origin_lon=-73.9665
)

In [255]:
def filter_exit_candidates(poi_table):
    df = poi_table.copy()

    # 转小写方便判断
    df["name_lc"] = df["name"].fillna("").astype(str).str.lower() if "name" in df.columns else ""
    df["barrier_lc"] = df["barrier"].fillna("").astype(str).str.lower() if "barrier" in df.columns else ""
    df["entrance_lc"] = df["entrance"].fillna("").astype(str).str.lower() if "entrance" in df.columns else ""
    df["display_name_lc"] = df["display_name"].fillna("").astype(str).str.lower()

    # 更严格一点：优先真实 gate / 有名字的入口
    mask = (
        (df["barrier_lc"] == "gate") |
        (df["name_lc"].str.contains("gate", na=False)) |
        (df["display_name_lc"].str.contains("gate", na=False))
    )

    out = df[mask].copy()
    return out

In [267]:
from shapely.geometry import shape
from shapely.geometry.base import BaseGeometry

def normalize_boundary_geometry(boundary_obj):
    """
    Accepts:
    - shapely Polygon / MultiPolygon
    - GeoDataFrame
    - GeoSeries
    - GeoJSON Feature
    - GeoJSON FeatureCollection
    Returns a shapely geometry
    """
    # 1) already shapely geometry
    if isinstance(boundary_obj, BaseGeometry):
        return boundary_obj

    # 2) GeoDataFrame
    if isinstance(boundary_obj, gpd.GeoDataFrame):
        return boundary_obj.geometry.iloc[0]

    # 3) GeoSeries
    if isinstance(boundary_obj, gpd.GeoSeries):
        return boundary_obj.iloc[0]

    # 4) dict-like GeoJSON
    if isinstance(boundary_obj, dict):
        geo_type = boundary_obj.get("type", "").lower()

        if geo_type == "feature":
            return shape(boundary_obj["geometry"])

        if geo_type == "featurecollection":
            features = boundary_obj.get("features", [])
            if not features:
                raise ValueError("FeatureCollection is empty.")
            return shape(features[0]["geometry"])

        # direct geometry dict
        return shape(boundary_obj)

    raise TypeError(f"Unsupported boundary object type: {type(boundary_obj)}")

In [269]:
def find_best_exit_by_route(origin_lat, origin_lon, poi_table, park_boundary, top_k=8, boundary_top_k=8):
    df = filter_exit_candidates(poi_table)

    if df.empty:
        return None, pd.DataFrame()

    # 加 boundary distance
    df = add_boundary_distance(df, park_boundary)

    # 先按“离边界近”筛一批
    df = df.sort_values("boundary_dist_m", ascending=True).head(boundary_top_k).copy()

    # 再按“离起点直线近”筛
    df["euclid_dist"] = ((df["lat"] - origin_lat) ** 2 + (df["lon"] - origin_lon) ** 2) ** 0.5
    df = df.sort_values(["euclid_dist", "boundary_dist_m"], ascending=[True, True]).head(top_k).copy()

    rows = []

    for idx, row in df.iterrows():
        try:
            r = route_from_gps_and_text(
                origin_lat=origin_lat,
                origin_lon=origin_lon,
                destination_query=row["display_name"],
                poi_table=poi_table
            )

            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "boundary_dist_m": float(row["boundary_dist_m"]),
                "route_length_m": float(r["route_stats"]["length_m"]),
                "gpx_path": r.get("gpx_path", None)
            })

        except Exception as e:
            rows.append({
                "idx": idx,
                "display_name": row["display_name"],
                "lat": row["lat"],
                "lon": row["lon"],
                "euclid_dist": float(row["euclid_dist"]),
                "boundary_dist_m": float(row["boundary_dist_m"]),
                "route_length_m": None,
                "gpx_path": None,
                "error": str(e)
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return None, out

    out = out.sort_values("route_length_m", ascending=True, na_position="last")

    valid = out[out["route_length_m"].notna()].copy()
    if valid.empty:
        return None, out

    best_idx = int(valid.iloc[0]["idx"])
    best_row = poi_table.loc[best_idx]

    return best_row, out

In [275]:
def add_boundary_distance(df, park_boundary):
    temp = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    temp_proj = temp.to_crs("EPSG:3857")

    boundary_geom = normalize_boundary_geometry(park_boundary)
    boundary_series = gpd.GeoSeries([boundary_geom], crs="EPSG:4326").to_crs("EPSG:3857")
    boundary_proj = boundary_series.iloc[0]

    temp["boundary_dist_m"] = temp_proj.geometry.apply(
        lambda p: p.distance(boundary_proj.boundary)
    )

    return temp

In [271]:
def route_one_step_routeaware(origin_lat, origin_lon, category, poi_table, park_boundary, top_k=8):
    if category == "gate":
        best_row, candidate_compare = find_best_exit_by_route(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            park_boundary=park_boundary,
            top_k=top_k,
            boundary_top_k=12
        )
    else:
        best_row, candidate_compare = find_best_destination_by_route(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            category=category,
            top_k=top_k
        )

    if best_row is None:
        return {
            "status": "reject",
            "message": f"No valid destinations found for category: {category}",
            "category": category,
            "candidate_compare": candidate_compare.to_dict(orient="records") if isinstance(candidate_compare, pd.DataFrame) else []
        }

    r = route_from_gps_and_text(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        destination_query=best_row["display_name"],
        poi_table=poi_table
    )

    if "status" not in r:
        r["status"] = "ok"

    r["smart_category"] = category
    r["selection_method"] = "shortest_route_within_category" if category != "gate" else "shortest_route_among_real_exits"
    r["smart_selected_display_name"] = best_row["display_name"]
    r["candidate_compare"] = candidate_compare.to_dict(orient="records")

    return r

In [273]:
def run_ai_trip_routeaware(origin_lat, origin_lon, user_request, poi_table, park_boundary, top_k=8):
    ai_plan = ai_plan_trip_steps(user_request)
    steps = ai_plan["steps"]

    all_results = []
    current_lat = origin_lat
    current_lon = origin_lon

    for step in steps:
        category = step["category"]

        if category == "unclear":
            all_results.append({
                "step_index": step["step_index"],
                "user_intent": step["user_intent"],
                "category": category,
                "status": "clarify",
                "reason": step["reason"]
            })
            continue

        r = route_one_step_routeaware(
            origin_lat=current_lat,
            origin_lon=current_lon,
            category=category,
            poi_table=poi_table,
            park_boundary=park_boundary,
            top_k=top_k
        )

        all_results.append({
            "step_index": step["step_index"],
            "user_intent": step["user_intent"],
            "category": category,
            "ai_reason": step["reason"],
            "route_result": r
        })

        if r.get("status") == "ok":
            dest = r["destination_best_match"]
            current_lat = float(dest["lat"])
            current_lon = float(dest["lon"])

    return {
        "user_request": user_request,
        "ai_plan": ai_plan,
        "steps": all_results
    }

In [277]:
print(type(park_boundary))
print(getattr(park_boundary, "geom_type", "no geom_type"))

<class 'geopandas.geodataframe.GeoDataFrame'>
0    Polygon
dtype: object


In [279]:
exit_best, exit_compare = find_best_exit_by_route(
    origin_lat=40.7812,
    origin_lon=-73.9665,
    poi_table=dest_gdf,
    park_boundary=park_boundary,
    top_k=8,
    boundary_top_k=12
)

exit_compare[["display_name", "boundary_dist_m", "euclid_dist", "route_length_m"]]

,display_name,boundary_dist_m,euclid_dist,route_length_m
5,Vanderbilt Gate | gate,5.576657,0.019157,2108.455218
0,gate,58.922314,0.007236,2500.227966
1,gate,51.687993,0.013345,2500.227966
2,gate,80.298379,0.014561,2500.227966
3,gate,214.778904,0.015717,2500.227966
4,gate,36.030424,0.017929,2500.227966
6,gate,107.504032,0.019520,2500.227966
7,gate,298.934316,0.019568,2500.227966


In [281]:
def route_one_step_routeaware(origin_lat, origin_lon, category, poi_table, park_boundary, top_k=8):
    """
    单步 routing:
    - gate: 用真实 exit 候选 + shortest route
    - 其他 category: 用 category 内 shortest route
    """
    if category == "gate":
        best_row, candidate_compare = find_best_exit_by_route(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            park_boundary=park_boundary,
            top_k=top_k,
            boundary_top_k=12
        )
        selection_method = "shortest_route_among_real_exits"
    else:
        best_row, candidate_compare = find_best_destination_by_route(
            origin_lat=origin_lat,
            origin_lon=origin_lon,
            poi_table=poi_table,
            category=category,
            top_k=top_k
        )
        selection_method = "shortest_route_within_category"

    if best_row is None:
        return {
            "status": "reject",
            "message": f"No valid destinations found for category: {category}",
            "category": category,
            "candidate_compare": candidate_compare.to_dict(orient="records") if isinstance(candidate_compare, pd.DataFrame) else []
        }

    r = route_from_gps_and_text(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        destination_query=best_row["display_name"],
        poi_table=poi_table
    )

    if "status" not in r:
        r["status"] = "ok"

    r["smart_category"] = category
    r["selection_method"] = selection_method
    r["smart_selected_display_name"] = best_row["display_name"]
    r["candidate_compare"] = candidate_compare.to_dict(orient="records")

    return r

In [283]:
def summarize_ai_trip(ai_trip_result):
    rows = []

    for step in ai_trip_result["steps"]:
        rr = step.get("route_result", {})

        rows.append({
            "step_index": step["step_index"],
            "user_intent": step["user_intent"],
            "ai_category": step["category"],
            "ai_reason": step.get("ai_reason", step.get("reason", "")),
            "status": rr.get("status", step.get("status", "unknown")),
            "selection_method": rr.get("selection_method", None),
            "selected_destination": rr.get("smart_selected_display_name", None),
            "route_length_m": rr.get("route_stats", {}).get("length_m", None)
        })

    return pd.DataFrame(rows).sort_values("step_index")

In [285]:
def show_ai_trip_on_folium(ai_trip_result, origin_lat, origin_lon):
    color_cycle = ["red", "blue", "green", "purple", "orange"]
    m = folium.Map(location=[origin_lat, origin_lon], zoom_start=14)

    # 起点
    folium.Marker(
        [origin_lat, origin_lon],
        popup="Start",
        icon=folium.Icon(color="black", icon="play")
    ).add_to(m)

    step_num = 0

    for step in ai_trip_result["steps"]:
        rr = step.get("route_result", {})
        if rr.get("status") != "ok":
            continue

        step_num += 1
        color = color_cycle[(step_num - 1) % len(color_cycle)]

        route_latlon = rr["route_latlon"]
        folium.PolyLine(
            route_latlon,
            color=color,
            weight=6,
            opacity=0.85,
            popup=f"Step {step_num}: {step['user_intent']}"
        ).add_to(m)

        dest = rr["destination_best_match"]
        popup_html = f"""
        <b>Step {step_num}</b><br>
        Intent: {step['user_intent']}<br>
        AI category: {step['category']}<br>
        AI reason: {step['ai_reason']}<br>
        Selection method: {rr.get('selection_method', '')}<br>
        Selected: {rr['smart_selected_display_name']}<br>
        Route length: {round(rr['route_stats']['length_m'], 1)} m
        """

        folium.Marker(
            [dest["lat"], dest["lon"]],
            popup=folium.Popup(popup_html, max_width=380),
            icon=folium.Icon(color=color)
        ).add_to(m)

    return m

In [287]:
ai_trip_demo_2 = run_ai_trip_routeaware(
    origin_lat=40.7812,
    origin_lon=-73.9665,
    user_request="find a viewpoint, then go to the bathroom, then leave the park",
    poi_table=dest_gdf,
    park_boundary=park_boundary,
    top_k=8
)

summarize_ai_trip(ai_trip_demo_2)

,step_index,user_intent,ai_category,ai_reason,status,selection_method,selected_destination,route_length_m
0,1,find a viewpoint,viewpoint,User is looking for a scenic spot to enjoy the...,ok,shortest_route_within_category,slipway | viewpoint,950.534138
1,2,go to the bathroom,toilets,User needs to use the restroom.,ok,shortest_route_within_category,toilets,902.825431
2,3,leave the park,gate,User wants to exit the park.,ok,shortest_route_among_real_exits,Vanderbilt Gate | gate,3616.334965


In [293]:
show_ai_trip_on_folium(
    ai_trip_demo_2,
    origin_lat=40.7811,
    origin_lon=-73.9665
)